# Trading App V2 Equity Meta-Model Backtesting Experiment

Equity-only clone of `trading_app_v2` for testing whether a stacked meta classifier can outperform simple averaging across feature-family classifiers.

Primary success metric: post-2021 out-of-sample trading performance, not ML classification metrics. The classification diagnostics are included only as secondary context.

Experiment design:
- 10B market-cap universe by default.
- No option ranker, no ThetaData refresh, no broker orders, no Streamlit app.
- Train one classifier per requested feature family on all available data through `2020-12-31`.
- For meta-model training only, score base family models on oracle-label dates to avoid unnecessary in-sample daily scoring.
- Train a second classifier using those family model predictions as features and the same collapsed oracle trade labels.
- For backtesting, separately score every post-2021 daily feature row and compare `stacked_meta` against `ensemble_mean` and `ensemble_rank_mean` with the same simple threshold rule: buy when long score is above `0.5`, sell when it is below `0.5`.


In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import json
import os
import pickle
import sys
from time import perf_counter
from types import SimpleNamespace

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 280)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "app").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def prefer_local_repo(package_name: str, repo_dir: Path) -> None:
    repo_dir = repo_dir.resolve()
    if not repo_dir.exists():
        return
    repo_text = str(repo_dir)
    sys.path[:] = [
        entry
        for entry in sys.path
        if str(Path(entry or ".").expanduser().resolve()) != repo_text
    ]
    sys.path.insert(0, repo_text)
    loaded = sys.modules.get(package_name)
    loaded_file = Path(str(getattr(loaded, "__file__", ""))).resolve() if loaded is not None else None
    if loaded_file is not None and repo_dir not in loaded_file.parents:
        for module_name in [name for name in sys.modules if name == package_name or name.startswith(f"{package_name}.")]:
            del sys.modules[module_name]


WORKSPACE_ROOT = REPO_ROOT.parent
prefer_local_repo("quant_orchestrator", WORKSPACE_ROOT / "quant-orchestrator")
prefer_local_repo("quant_warehouse", WORKSPACE_ROOT / "quant-warehouse")
prefer_local_repo("fmpsdk", WORKSPACE_ROOT / "fmpsdk")

from app.quant_warehouse_storage import ensure_quant_warehouse_storage
from app.trading_app_v2_runtime import DEFAULT_STRATEGY_SOURCES, default_paths

paths = default_paths(REPO_ROOT)
paths.artifact_root.mkdir(parents=True, exist_ok=True)
META_ARTIFACT_DIR = paths.artifact_root / "equity_meta_model_10b"
META_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"repo_root": str(REPO_ROOT), "artifact_dir": str(META_ARTIFACT_DIR)})


## Environment


In [ ]:
storage = ensure_quant_warehouse_storage()
display(storage.as_dict())


def optional_date(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text or text.lower() in {"none", "null", "nat"}:
        return None
    return text


## Run Configuration


In [ ]:
RUN_FMP_REFRESH = False  # Use cached warehouse data by default; set True only for an intentional FMP refresh.
RUN_EQUITY_META_EXPERIMENT = True
RUN_BACKTESTS = True
RUN_ANCHORED_WFO = True
REUSE_ANCHORED_WFO_FOLDS = False
ANCHORED_WFO_TOP_K_VALUES = (None, 5, 10, 20, 40)
INCLUDE_FAMILY_STRATEGY_WFO = True
INCLUDE_FAMILY_SCORE_RANK_FEATURES = True
INCLUDE_RANK_MEAN_ENSEMBLE = True
ENSEMBLE_STRATEGY_SOURCES = ("ensemble_mean", "ensemble_rank_mean") if INCLUDE_RANK_MEAN_ENSEMBLE else ("ensemble_mean",)
ANCHORED_WFO_OUTPUT_NAME = (
    "anchored_wfo_family_strategies_rank_features"
    if INCLUDE_FAMILY_STRATEGY_WFO and INCLUDE_FAMILY_SCORE_RANK_FEATURES
    else ("anchored_wfo_family_strategies" if INCLUDE_FAMILY_STRATEGY_WFO else "anchored_wfo")
)
INCLUDE_TECHNICAL_FEATURE_FAMILIES = True
TECHNICAL_STRATEGY_SOURCES = (
    "fmp.technical_candles",
    "fmp.technical_cycles",
    "fmp.technical_math",
    "fmp.technical_momentum",
    "fmp.technical_overlap",
    "fmp.technical_performance",
)
EXPERIMENT_STRATEGY_SOURCES = tuple(
    dict.fromkeys(
        (
            *DEFAULT_STRATEGY_SOURCES,
            *(TECHNICAL_STRATEGY_SOURCES if INCLUDE_TECHNICAL_FEATURE_FAMILIES else ()),
        )
    )
)

DATA_START = "1900-01-01"
DATA_END = ""  # Empty means latest available downstream data.
MIN_MARKET_CAP = 10_000_000_000
BASE_TRAIN_END = "2020-12-31"
META_OOS_START = "2021-01-01"
SCORE_THRESHOLD = 0.50
SIMPLE_RULE_COST_BPS = 5.5
REUSE_DAILY_SCORE_CHUNKS = False  # legacy derived-score pickle chunks; prefer family-score parquet cache below
REUSE_FAMILY_SCORE_CACHE = True
REBUILD_FAMILY_SCORE_CACHE = False
RUN_SYMBOL_LEVEL_BACKTESTING_PY = True
REUSE_SYMBOL_LEVEL_BACKTESTS = False
SYMBOL_BACKTEST_CHECKPOINT_EVERY = 25
SAVE_LARGE_INTERMEDIATE_CSVS = False
REBUILD_FEATURE_FAMILY_CACHE = False
REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES = True
RUN_REGIME_YEAR_FEATURE = True
REGIME_YEAR_FEATURE = "regime_calendar_year"
INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS = True
EVENT_LABEL_FAMILIES = (
    "congress",
    "insider",
    "analyst_rating",
    "analyst_estimate",
    "price_target",
    "institutional",
    "capital_action",
    "dividend",
    "split",
    "earnings",
)
EVENT_LABEL_ALIGNMENT_TOLERANCE_DAYS = 0
EVENT_LABEL_INCLUDE_HISTORICAL = True

REFRESH_MISSING_FMP_INCLUDE_MACRO = True
REFRESH_MISSING_FMP_INCLUDE_PRICES = True
REFRESH_MISSING_FMP_STALENESS_DAYS = 60
REFRESH_MISSING_FMP_SKIP_RECENT_HOURS = 24.0
REFRESH_MISSING_FMP_MAX_WORKERS = 8

BASE_RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": 16,
    "max_features": "sqrt",
    "n_bins": 128,
    "n_streams": 8,
}
META_RF_PARAMS = {
    "n_estimators": 500,
    "max_depth": 8,
    "max_features": "sqrt",
    "n_bins": 128,
    "n_streams": 8,
}
RANDOM_SEED = 20260707

RUN_ARTIFACT_KEY = f"mcap_{int(MIN_MARKET_CAP)}_train_{BASE_TRAIN_END}_seed_{RANDOM_SEED}"
RUN_ARTIFACT_DIR = META_ARTIFACT_DIR / RUN_ARTIFACT_KEY
RUN_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(
    {
        "universe": "10B market cap",
        "min_market_cap": MIN_MARKET_CAP,
        "base_train_end": BASE_TRAIN_END,
        "meta_oos_start": META_OOS_START,
        "score_threshold": SCORE_THRESHOLD,
        "simple_rule_cost_bps": SIMPLE_RULE_COST_BPS,
        "reuse_daily_score_chunks": REUSE_DAILY_SCORE_CHUNKS,
        "reuse_family_score_cache": REUSE_FAMILY_SCORE_CACHE,
        "rebuild_family_score_cache": REBUILD_FAMILY_SCORE_CACHE,
        "run_symbol_level_backtesting_py": RUN_SYMBOL_LEVEL_BACKTESTING_PY,
        "reuse_symbol_level_backtests": REUSE_SYMBOL_LEVEL_BACKTESTS,
        "save_large_intermediate_csvs": SAVE_LARGE_INTERMEDIATE_CSVS,
        "rebuild_feature_family_cache": REBUILD_FEATURE_FAMILY_CACHE,
        "run_regime_year_feature": RUN_REGIME_YEAR_FEATURE,
        "regime_year_feature": REGIME_YEAR_FEATURE if RUN_REGIME_YEAR_FEATURE else None,
        "include_event_labels_in_oracle_labels": INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS,
        "event_label_families": EVENT_LABEL_FAMILIES if INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS else (),
        "event_label_alignment_tolerance_days": EVENT_LABEL_ALIGNMENT_TOLERANCE_DAYS,
        "run_backtests": RUN_BACKTESTS,
        "run_anchored_wfo": RUN_ANCHORED_WFO,
        "anchored_wfo_top_k_values": ANCHORED_WFO_TOP_K_VALUES,
        "include_family_strategy_wfo": INCLUDE_FAMILY_STRATEGY_WFO,
        "include_family_score_rank_features": INCLUDE_FAMILY_SCORE_RANK_FEATURES,
        "include_rank_mean_ensemble": INCLUDE_RANK_MEAN_ENSEMBLE,
        "ensemble_strategy_sources": ENSEMBLE_STRATEGY_SOURCES,
        "anchored_wfo_output_name": ANCHORED_WFO_OUTPUT_NAME,
        "include_technical_feature_families": INCLUDE_TECHNICAL_FEATURE_FAMILIES,
        "technical_strategy_sources": TECHNICAL_STRATEGY_SOURCES if INCLUDE_TECHNICAL_FEATURE_FAMILIES else (),
        "experiment_strategy_sources": EXPERIMENT_STRATEGY_SOURCES,
        "artifact_dir": str(RUN_ARTIFACT_DIR),
    }
)


## FMP Historical Refresh


In [ ]:
from quant_warehouse.migrate.backfill_missing_fmp import backfill_missing_fmp_historical
from quant_warehouse.research_tools.feature_family_eval import FamilyEvaluationConfig, screen_fmp_equity_universe
from quant_warehouse.warehouse.api import Warehouse

refresh_warehouse = Warehouse()
refresh_feature_config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
)

screened_equity_symbols, raw_fmp_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(
    refresh_feature_config,
    warehouse=refresh_warehouse,
)
print(
    f"FMP universe source={universe_source}; "
    f"eligible_symbols={len(screened_equity_symbols):,}; "
    f"min_market_cap={MIN_MARKET_CAP:,}"
)
display(pd.DataFrame({"symbol": list(screened_equity_symbols)}))
display(universe_eligibility.head(100))


def notebook_refresh_log(message: str) -> None:
    print(f"[fmp-refresh] {message}", flush=True)


fmp_refresh_summary = {"status": "skipped_by_config", "equity_symbols": list(screened_equity_symbols), "etf_symbols": []}
if RUN_FMP_REFRESH:
    fmp_refresh_summary = backfill_missing_fmp_historical(
        warehouse=refresh_warehouse,
        equity_provider="fmp",
        etf_provider="fmp",
        equity_symbols=screened_equity_symbols,
        etf_symbols=(),
        include_macro=REFRESH_MISSING_FMP_INCLUDE_MACRO,
        include_prices=REFRESH_MISSING_FMP_INCLUDE_PRICES,
        staleness_days=REFRESH_MISSING_FMP_STALENESS_DAYS,
        skip_recent_hours=REFRESH_MISSING_FMP_SKIP_RECENT_HOURS,
        max_workers=REFRESH_MISSING_FMP_MAX_WORKERS,
        progress_logger=notebook_refresh_log,
    )
else:
    print("FMP refresh skipped because RUN_FMP_REFRESH=False")

fmp_refresh_summary_path = RUN_ARTIFACT_DIR / "fmp_refresh_summary.json"
fmp_refresh_summary_path.write_text(json.dumps(fmp_refresh_summary, indent=2, default=str), encoding="utf-8")
display(pd.DataFrame([{
    "equity_symbols": len(fmp_refresh_summary.get("equity_symbols", [])),
    "equity_price_total": fmp_refresh_summary.get("equity_prices", {}).get("total"),
    "equity_price_updated": fmp_refresh_summary.get("equity_prices", {}).get("updated"),
    "equity_fundamental_total": fmp_refresh_summary.get("equity", {}).get("total"),
    "equity_fundamental_updated": fmp_refresh_summary.get("equity", {}).get("updated"),
}]))


## Target Engineering


In [ ]:
from quant_warehouse.research_tools import BinaryTargetConfig, build_event_target_panel, load_fmp_event_pairs
from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EVENT_PAIR_TAXONOMY, EventPairStore
from quant_warehouse.warehouse.api import Warehouse
from quant_orchestrator.research_tools.ml_trading_experiment import _build_oracle_trade_label_rows_sparse

if "screened_equity_symbols" not in globals():
    raise RuntimeError("Run the FMP Historical Refresh cell before target engineering.")

target_warehouse = Warehouse()
target_config = BinaryTargetConfig(
    provider="fmp",
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
    event_families=EVENT_LABEL_FAMILIES,
    event_alignment_tolerance_days=EVENT_LABEL_ALIGNMENT_TOLERANCE_DAYS,
    oracle_trade_k_by_frequency={"YE": tuple(range(1, 13))},
)

ORACLE_LABEL_CACHE_DIR = RUN_ARTIFACT_DIR / "oracle_labels"
ORACLE_LABEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
ORACLE_LABEL_ROWS_PATH = ORACLE_LABEL_CACHE_DIR / "oracle_label_rows.parquet"
ORACLE_LABEL_DIAGNOSTICS_PATH = ORACLE_LABEL_CACHE_DIR / "oracle_label_diagnostics.parquet"
ORACLE_TRADE_WINDOWS_UNIQUE_PATH = ORACLE_LABEL_CACHE_DIR / "oracle_trade_windows_unique.parquet"
EVENT_LABEL_ROWS_PATH = ORACLE_LABEL_CACHE_DIR / "event_label_rows.parquet"
EVENT_LABEL_DIAGNOSTICS_PATH = ORACLE_LABEL_CACHE_DIR / "event_label_diagnostics.parquet"
ORACLE_LABEL_MANIFEST_PATH = ORACLE_LABEL_CACHE_DIR / "manifest.json"

_ENABLED_EVENT_TAXONOMY = {family: EVENT_PAIR_TAXONOMY[family] for family in EVENT_LABEL_FAMILIES}
_BULLISH_EVENT_TYPES = {pair["positive"] for pair in _ENABLED_EVENT_TAXONOMY.values()}
_BEARISH_EVENT_TYPES = {pair["negative"] for pair in _ENABLED_EVENT_TAXONOMY.values()}
_EVENT_TYPE_TO_FAMILY = {
    pair[side]: family
    for family, pair in _ENABLED_EVENT_TAXONOMY.items()
    for side in ("positive", "negative")
}


def _target_price_base_panel(symbols: tuple[str, ...]) -> pd.DataFrame:
    frames = []
    for raw_symbol in symbols:
        symbol = str(raw_symbol).strip().upper()
        if not symbol:
            continue
        prices = target_warehouse.read_prices(
            symbol,
            provider=target_config.provider,
            start=target_config.start_date,
            end=target_config.end_date,
        )
        if prices is None or prices.empty:
            continue
        raw_dates = prices.index if "date" not in prices.columns else prices["date"]
        dates = pd.Series(pd.to_datetime(raw_dates, errors="coerce")).dt.normalize()
        frame = pd.DataFrame({"symbol": symbol, "date": dates}).dropna(subset=["date"])
        if not frame.empty:
            frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=["symbol", "date"])
    out = pd.concat(frames, ignore_index=True).drop_duplicates()
    out["symbol"] = out["symbol"].astype(str).str.upper()
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize()
    return out.sort_values(["symbol", "date"]).reset_index(drop=True)


def _event_target_columns_for_labels(event_target_panel: pd.DataFrame) -> tuple[list[str], list[str]]:
    bullish_cols = [f"target_event_on__{event_type}" for event_type in sorted(_BULLISH_EVENT_TYPES) if f"target_event_on__{event_type}" in event_target_panel.columns]
    bearish_cols = [f"target_event_on__{event_type}" for event_type in sorted(_BEARISH_EVENT_TYPES) if f"target_event_on__{event_type}" in event_target_panel.columns]
    return bullish_cols, bearish_cols


def _event_labels_from_target_panel(event_target_panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if event_target_panel.empty:
        empty = pd.DataFrame(columns=["symbol", "date", "collapsed_label", "label_source", "event_type", "event_family"])
        diagnostics = pd.DataFrame(
            [
                {
                    "source": "event_labels",
                    "candidate_rows": 0,
                    "used_rows": 0,
                    "dropped_rows": 0,
                    "note": "event target panel empty",
                }
            ]
        )
        return empty, diagnostics

    bullish_cols, bearish_cols = _event_target_columns_for_labels(event_target_panel)
    rows = []
    for col in bullish_cols:
        event_type = col.removeprefix("target_event_on__")
        mask = pd.to_numeric(event_target_panel[col], errors="coerce").fillna(0).gt(0)
        part = event_target_panel.loc[mask, ["symbol", "date"]].copy()
        if part.empty:
            continue
        part["collapsed_label"] = "oracle_long"
        part["label_source"] = f"event_{event_type}"
        part["event_type"] = event_type
        part["event_family"] = _EVENT_TYPE_TO_FAMILY.get(event_type, event_type.split("_", 1)[0])
        rows.append(part)
    for col in bearish_cols:
        event_type = col.removeprefix("target_event_on__")
        mask = pd.to_numeric(event_target_panel[col], errors="coerce").fillna(0).gt(0)
        part = event_target_panel.loc[mask, ["symbol", "date"]].copy()
        if part.empty:
            continue
        part["collapsed_label"] = "oracle_short"
        part["label_source"] = f"event_{event_type}"
        part["event_type"] = event_type
        part["event_family"] = _EVENT_TYPE_TO_FAMILY.get(event_type, event_type.split("_", 1)[0])
        rows.append(part)

    labels = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["symbol", "date", "collapsed_label", "label_source", "event_type", "event_family"])
    if not labels.empty:
        labels["symbol"] = labels["symbol"].astype(str).str.upper()
        labels["date"] = pd.to_datetime(labels["date"], errors="coerce").dt.normalize()
        labels = labels.dropna(subset=["symbol", "date", "collapsed_label"]).drop_duplicates()
        side_counts = labels.groupby(["event_type", "collapsed_label"]).size().reset_index(name="used_rows")
    else:
        side_counts = pd.DataFrame(columns=["event_type", "collapsed_label", "used_rows"])

    candidate_rows = int(sum(pd.to_numeric(event_target_panel[col], errors="coerce").fillna(0).gt(0).sum() for col in [*bullish_cols, *bearish_cols]))
    diagnostics = pd.DataFrame(
        [
            {
                "source": "event_labels",
                "candidate_rows": candidate_rows,
                "used_rows": len(labels),
                "dropped_rows": max(0, candidate_rows - len(labels)),
                "note": "event rows mapped into oracle_long/oracle_short labels",
            }
        ]
    )
    if not side_counts.empty:
        diagnostics = pd.concat(
            [
                diagnostics,
                side_counts.assign(
                    source=lambda frame: "event_" + frame["event_type"].astype(str),
                    candidate_rows=lambda frame: frame["used_rows"],
                    dropped_rows=0,
                    note="event type mapped into binary oracle label space",
                )[diagnostics.columns],
            ],
            ignore_index=True,
            sort=False,
        )
    return labels.sort_values(["date", "symbol", "collapsed_label", "label_source"]).reset_index(drop=True), diagnostics


def _combine_oracle_and_event_labels(oracle_rows: pd.DataFrame, event_rows: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    frames = []
    if oracle_rows is not None and not oracle_rows.empty:
        frames.append(oracle_rows.copy())
    if event_rows is not None and not event_rows.empty:
        frames.append(event_rows[["symbol", "date", "collapsed_label", "label_source"]].copy())
    if not frames:
        empty = pd.DataFrame(columns=["symbol", "date", "collapsed_label", "label_source"])
        return empty, pd.DataFrame()
    labels = pd.concat(frames, ignore_index=True, sort=False)
    labels["symbol"] = labels["symbol"].astype(str).str.upper()
    labels["date"] = pd.to_datetime(labels["date"], errors="coerce").dt.normalize()
    labels = labels.dropna(subset=["symbol", "date", "collapsed_label"])
    before_conflict = len(labels)
    side_counts = labels.groupby(["symbol", "date"])["collapsed_label"].nunique()
    conflicting_keys = side_counts.loc[side_counts.gt(1)].index
    if len(conflicting_keys):
        conflict_frame = pd.DataFrame(list(conflicting_keys), columns=["symbol", "date"])
        labels = labels.merge(conflict_frame.assign(_conflict=True), on=["symbol", "date"], how="left")
        labels = labels.loc[~labels["_conflict"].fillna(False)].drop(columns=["_conflict"])
    labels = (
        labels.assign(label_source=labels["label_source"].fillna("unknown").astype(str))
        .groupby(["symbol", "date", "collapsed_label"], as_index=False)
        .agg(label_source=("label_source", lambda values: "|".join(dict.fromkeys(sorted(values)))))
        .sort_values(["date", "symbol", "collapsed_label", "label_source"])
        .reset_index(drop=True)
    )
    diagnostics = pd.DataFrame(
        [
            {
                "source": "combined_labels",
                "candidate_rows": before_conflict,
                "used_rows": len(labels),
                "dropped_rows": before_conflict - len(labels),
                "note": "symbol/date conflicts with both long and short labels dropped",
            }
        ]
    )
    return labels, diagnostics


def _oracle_label_cache_matches() -> bool:
    if (
        not ORACLE_LABEL_ROWS_PATH.exists()
        or not ORACLE_LABEL_DIAGNOSTICS_PATH.exists()
        or not ORACLE_LABEL_MANIFEST_PATH.exists()
        or not ORACLE_TRADE_WINDOWS_UNIQUE_PATH.exists()
    ):
        return False
    try:
        manifest = json.loads(ORACLE_LABEL_MANIFEST_PATH.read_text(encoding="utf-8"))
    except Exception:
        return False
    return (
        manifest.get("min_market_cap") == int(MIN_MARKET_CAP)
        and manifest.get("symbols") == list(map(str, screened_equity_symbols))
        and manifest.get("oracle_trade_k_by_frequency") == {"YE": list(range(1, 13))}
        and manifest.get("start_date") == DATA_START
        and manifest.get("end_date") == optional_date(DATA_END)
        and manifest.get("include_event_labels_in_oracle_labels") == bool(INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS)
        and manifest.get("event_label_families") == list(EVENT_LABEL_FAMILIES if INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS else ())
        and manifest.get("event_label_alignment_tolerance_days") == int(EVENT_LABEL_ALIGNMENT_TOLERANCE_DAYS)
        and manifest.get("event_label_include_historical") == bool(EVENT_LABEL_INCLUDE_HISTORICAL)
        and manifest.get("unique_trade_windows") is True
    )


if _oracle_label_cache_matches():
    oracle_label_rows = pd.read_parquet(ORACLE_LABEL_ROWS_PATH)
    oracle_label_diagnostics = pd.read_parquet(ORACLE_LABEL_DIAGNOSTICS_PATH)
    oracle_trade_windows_unique = pd.read_parquet(ORACLE_TRADE_WINDOWS_UNIQUE_PATH)
    event_label_rows = pd.read_parquet(EVENT_LABEL_ROWS_PATH) if EVENT_LABEL_ROWS_PATH.exists() else pd.DataFrame()
    event_label_diagnostics = pd.read_parquet(EVENT_LABEL_DIAGNOSTICS_PATH) if EVENT_LABEL_DIAGNOSTICS_PATH.exists() else pd.DataFrame()
    manifest = json.loads(ORACLE_LABEL_MANIFEST_PATH.read_text(encoding="utf-8"))
    oracle_seconds = float(manifest.get("oracle_seconds", 0.0))
    event_seconds = float(manifest.get("event_seconds", 0.0))
    print(f"Using cached oracle+event labels from {ORACLE_LABEL_CACHE_DIR}")
else:
    (
        oracle_only_label_rows,
        oracle_only_label_diagnostics,
        oracle_seconds,
        oracle_trade_windows_unique,
    ) = _build_oracle_trade_label_rows_sparse(
        screened_equity_symbols,
        target_config,
        warehouse=target_warehouse,
    )
    event_seconds = 0.0
    event_label_rows = pd.DataFrame(columns=["symbol", "date", "collapsed_label", "label_source", "event_type", "event_family"])
    event_label_diagnostics = pd.DataFrame(
        [
            {
                "source": "event_labels",
                "candidate_rows": 0,
                "used_rows": 0,
                "dropped_rows": 0,
                "note": "event labels disabled",
            }
        ]
    )
    event_load_diagnostics = pd.DataFrame()
    if INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS:
        event_store = EventPairStore(backend=target_warehouse.backend, catalog=target_warehouse.catalog)
        events, event_load_diagnostics, event_seconds = load_fmp_event_pairs(
            screened_equity_symbols,
            target_config,
            event_store=event_store,
            include_historical=EVENT_LABEL_INCLUDE_HISTORICAL,
        )
        target_base_panel = _target_price_base_panel(tuple(screened_equity_symbols))
        event_target_panel, _event_target_metadata = build_event_target_panel(target_base_panel, events, target_config)
        event_label_rows, event_label_diagnostics = _event_labels_from_target_panel(event_target_panel)
        event_label_diagnostics = pd.concat(
            [
                event_label_diagnostics,
                pd.DataFrame(
                    [
                        {
                            "source": "event_pair_load",
                            "candidate_rows": len(events),
                            "used_rows": len(event_label_rows),
                            "dropped_rows": max(0, len(events) - len(event_label_rows)),
                            "note": f"loaded event pairs for families={EVENT_LABEL_FAMILIES}",
                        }
                    ]
                ),
            ],
            ignore_index=True,
            sort=False,
        )
    oracle_label_rows, combined_label_diagnostics = _combine_oracle_and_event_labels(oracle_only_label_rows, event_label_rows)
    oracle_label_diagnostics = pd.concat(
        [oracle_only_label_diagnostics, event_label_diagnostics, combined_label_diagnostics],
        ignore_index=True,
        sort=False,
    )
    oracle_label_rows.to_parquet(ORACLE_LABEL_ROWS_PATH, index=False)
    oracle_label_diagnostics.to_parquet(ORACLE_LABEL_DIAGNOSTICS_PATH, index=False)
    oracle_trade_windows_unique.to_parquet(ORACLE_TRADE_WINDOWS_UNIQUE_PATH, index=False)
    event_label_rows.to_parquet(EVENT_LABEL_ROWS_PATH, index=False)
    event_label_diagnostics.to_parquet(EVENT_LABEL_DIAGNOSTICS_PATH, index=False)
    ORACLE_LABEL_MANIFEST_PATH.write_text(
        json.dumps(
            {
                "min_market_cap": int(MIN_MARKET_CAP),
                "symbols": list(map(str, screened_equity_symbols)),
                "oracle_trade_k_by_frequency": {"YE": list(range(1, 13))},
                "start_date": DATA_START,
                "end_date": optional_date(DATA_END),
                "include_event_labels_in_oracle_labels": bool(INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS),
                "event_label_families": list(EVENT_LABEL_FAMILIES if INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS else ()),
                "event_label_alignment_tolerance_days": int(EVENT_LABEL_ALIGNMENT_TOLERANCE_DAYS),
                "event_label_include_historical": bool(EVENT_LABEL_INCLUDE_HISTORICAL),
                "unique_trade_windows": True,
                "oracle_unique_trade_windows": int(len(oracle_trade_windows_unique)),
                "oracle_seconds": oracle_seconds,
                "event_seconds": event_seconds,
            },
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )

oracle_label_rows["symbol"] = oracle_label_rows["symbol"].astype(str).str.upper()
oracle_label_rows["date"] = pd.to_datetime(oracle_label_rows["date"], errors="coerce").dt.normalize()
if not event_label_rows.empty:
    event_label_rows["symbol"] = event_label_rows["symbol"].astype(str).str.upper()
    event_label_rows["date"] = pd.to_datetime(event_label_rows["date"], errors="coerce").dt.normalize()

label_source_summary = (
    oracle_label_rows.groupby(["label_source", "collapsed_label"]).size().reset_index(name="rows")
    if not oracle_label_rows.empty and {"label_source", "collapsed_label"}.issubset(oracle_label_rows.columns)
    else pd.DataFrame()
)
print(
    {
        "oracle_plus_event_label_rows": len(oracle_label_rows),
        "oracle_unique_trade_windows": len(oracle_trade_windows_unique) if "oracle_trade_windows_unique" in globals() else None,
        "oracle_classes": sorted(oracle_label_rows["collapsed_label"].dropna().unique()) if not oracle_label_rows.empty else [],
        "oracle_seconds": round(oracle_seconds, 3),
        "event_seconds": round(event_seconds, 3),
        "event_label_rows": len(event_label_rows),
        "event_label_families": EVENT_LABEL_FAMILIES if INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS else (),
        "oracle_label_cache_dir": str(ORACLE_LABEL_CACHE_DIR),
    }
)
display(oracle_label_diagnostics)
display(label_source_summary.sort_values(["label_source", "collapsed_label"]) if not label_source_summary.empty else label_source_summary)
display(oracle_label_rows.head(20))


## Feature Engineering


In [ ]:
import gc
import re
from quant_warehouse.research_tools import FamilyEvaluationConfig
from quant_warehouse.research_tools.feature_family_eval import (
    _add_cross_symbol_context_features,
    _add_macro_context_features,
    _add_time_calendar_features,
    _build_symbol_fundamental_panel,
    _slice_frame,
    FeatureSpec,
    GROUP_PE_FEATURES,
)
from quant_warehouse.warehouse.api import Warehouse
from quant_warehouse.platforms.data_providers.fmp.feature_engineering.technical import build_price_technical_features
from quant_warehouse.platforms.data_providers.fmp.feature_engineering.ta_classic_technical import (
    TA_CLASSIC_FAMILY_PREFIXES,
    TaIndicatorSpec,
    _candlestick_specs,
    _compute_indicator,
    _feature_column_name,
    _import_pandas_ta_classic,
    _prepare_price_frame,
    _suppress_pandas_ta_classic_row_warnings,
    _to_built_feature_set,
    _usable_feature_cols,
)

if "screened_equity_symbols" not in globals():
    raise RuntimeError("Run the FMP Historical Refresh cell before feature engineering.")

engineering_warehouse = Warehouse()
feature_config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
)

REQUESTED_FAMILY_PAIRS = tuple(
    (name.split(".", 1)[0], name.split(".", 1)[1])
    for name in EXPERIMENT_STRATEGY_SOURCES
)
REQUESTED_FAMILY_SET = set(REQUESTED_FAMILY_PAIRS)
TECHNICAL_FAMILY_PAIRS = {
    *(("fmp", family) for family in TA_CLASSIC_FAMILY_PREFIXES),
}
REQUESTED_TECHNICAL_FAMILY_PAIRS = tuple(pair for pair in REQUESTED_FAMILY_PAIRS if pair in TECHNICAL_FAMILY_PAIRS)
CONTEXT_FAMILIES = {
    ("fmp", "time_calendar"),
    ("fmp", "economic_indicators"),
    ("fmp", "treasury_rates"),
    ("fmp", "sector_performance"),
    ("fmp", "industry_performance"),
    ("fmp", "sector_pe"),
    ("fmp", "industry_pe"),
}
PER_SYMBOL_FAMILIES = tuple(pair for pair in REQUESTED_FAMILY_PAIRS if pair not in CONTEXT_FAMILIES)
REQUESTED_CONTEXT_FAMILIES = tuple(pair for pair in REQUESTED_FAMILY_PAIRS if pair in CONTEXT_FAMILIES)

FEATURE_FAMILY_CACHE_DIR = RUN_ARTIFACT_DIR / "feature_family_panels"
FEATURE_FAMILY_CACHE_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_FAMILY_MANIFEST_PATH = FEATURE_FAMILY_CACHE_DIR / "manifest.json"


def _feature_family_file_name(source: str, family: str, suffix: str) -> str:
    token = re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{source}.{family}")
    return f"{token}.{suffix}"


def _feature_family_paths(source: str, family: str) -> tuple[Path, Path]:
    return (
        FEATURE_FAMILY_CACHE_DIR / _feature_family_file_name(source, family, "parquet"),
        FEATURE_FAMILY_CACHE_DIR / _feature_family_file_name(source, family, "metadata.parquet"),
    )


def _downcast_numeric_columns(frame: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    out = frame.copy()
    # Larger universes can expose duplicate columns after endpoint joins.
    out = out.loc[:, ~out.columns.duplicated()]
    feature_cols = list(dict.fromkeys(feature_cols))
    out["symbol"] = out["symbol"].astype(str).str.upper()
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize()
    float32_max = np.finfo(np.float32).max
    for col in feature_cols:
        out[col] = (
            pd.to_numeric(out[col], errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .clip(lower=-float32_max, upper=float32_max)
            .astype("float32")
        )
    return out.dropna(subset=["symbol", "date"]).reset_index(drop=True)


def _metadata_frame(specs) -> pd.DataFrame:
    if not specs:
        return pd.DataFrame(columns=["feature", "family", "source", "source_column", "expected_direction"])
    return (
        pd.DataFrame([spec.__dict__ for spec in specs])
        .drop_duplicates()
        .sort_values(["source", "family", "feature"])
        .reset_index(drop=True)
    )


def _technical_feature_specs(family: str, feature_cols: list[str]) -> list[FeatureSpec]:
    return [
        FeatureSpec(
            feature=str(feature),
            family=str(family),
            source="fmp",
            source_column=f"prices.{family}.{feature}",
            expected_direction="neutral",
        )
        for feature in feature_cols
    ]


def _built_feature_set_to_panel(symbol: str, feature_set) -> tuple[pd.DataFrame, list[str]]:
    feature_cols = list(getattr(feature_set, "feature_cols", []) or [])
    raw = getattr(feature_set, "df", pd.DataFrame())
    if raw is None or raw.empty or not feature_cols:
        return pd.DataFrame(columns=["symbol", "date"]), []
    out = raw.copy().reset_index()
    if "date" not in out.columns:
        for candidate in ("level_0", "index"):
            if candidate in out.columns:
                out = out.rename(columns={candidate: "date"})
                break
    if "symbol" not in out.columns:
        out["symbol"] = str(symbol).strip().upper()
    feature_cols = [col for col in feature_cols if col in out.columns]
    if "date" not in out.columns or not feature_cols:
        return pd.DataFrame(columns=["symbol", "date"]), []
    return _downcast_numeric_columns(out[["symbol", "date", *feature_cols]], feature_cols), feature_cols


def _curated_ta_indicator_specs(ta) -> dict[str, tuple[TaIndicatorSpec, ...]]:
    return {
        "technical_candles": tuple(_candlestick_specs(ta)),
        "technical_cycles": (
            TaIndicatorSpec("ebsw_40_10", "ebsw", ("close",), {"length": 40, "bars": 10}, min_rows=40),
            TaIndicatorSpec("ht_dcperiod", "ht_dcperiod", ("close",)),
            TaIndicatorSpec("ht_dcphase", "ht_dcphase", ("close",)),
            TaIndicatorSpec("ht_phasor", "ht_phasor", ("close",)),
            TaIndicatorSpec("ht_sine", "ht_sine", ("close",)),
            TaIndicatorSpec("ht_trendmode", "ht_trendmode", ("close",)),
        ),
        "technical_math": (
            TaIndicatorSpec("zscore_20", "zscore", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("zscore_63", "zscore", ("close",), {"length": 63}, min_rows=63),
            TaIndicatorSpec("entropy_20", "entropy", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("stdev_20", "stdev", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("variance_20", "variance", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("skew_20", "skew", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("kurtosis_20", "kurtosis", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("slope_20", "slope", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("mad_20", "mad", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("median_20", "median", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("quantile_20_25", "quantile", ("close",), {"length": 20, "q": 0.25}, min_rows=20),
            TaIndicatorSpec("quantile_20_50", "quantile", ("close",), {"length": 20, "q": 0.5}, min_rows=20),
            TaIndicatorSpec("quantile_20_75", "quantile", ("close",), {"length": 20, "q": 0.75}, min_rows=20),
            TaIndicatorSpec("linregslope_20", "linregslope", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("tos_stdevall_20", "tos_stdevall", ("close",), {"length": 20}, min_rows=20),
        ),
        "technical_momentum": (
            TaIndicatorSpec("rsi_14", "rsi", ("close",), {"length": 14}, min_rows=14),
            TaIndicatorSpec("macd", "macd", ("close",), min_rows=26),
            TaIndicatorSpec("stoch", "stoch", ("high", "low", "close"), min_rows=14),
            TaIndicatorSpec("stochrsi", "stochrsi", ("close",), {"length": 14, "rsi_length": 14}, min_rows=14),
            TaIndicatorSpec("cci_20", "cci", ("high", "low", "close"), {"length": 20}, min_rows=20),
            TaIndicatorSpec("roc_10", "roc", ("close",), {"length": 10}, min_rows=10),
            TaIndicatorSpec("mom_10", "mom", ("close",), {"length": 10}, min_rows=10),
            TaIndicatorSpec("willr_14", "willr", ("high", "low", "close"), {"length": 14}, min_rows=14),
            TaIndicatorSpec("ppo", "ppo", ("close",), min_rows=26),
            TaIndicatorSpec("cmo_14", "cmo", ("close",), {"length": 14}, min_rows=14),
            TaIndicatorSpec("bop", "bop", ("open", "high", "low", "close")),
            TaIndicatorSpec("ao", "ao", ("high", "low"), min_rows=34),
            TaIndicatorSpec("adx_14", "adx", ("high", "low", "close"), {"length": 14}, min_rows=14),
            TaIndicatorSpec("atr_14", "atr", ("high", "low", "close"), {"length": 14}, min_rows=14),
            TaIndicatorSpec("mfi_14", "mfi", ("high", "low", "close", "volume"), {"length": 14}, min_rows=14),
            TaIndicatorSpec("obv", "obv", ("close", "volume")),
            TaIndicatorSpec("adosc", "adosc", ("high", "low", "close", "volume"), min_rows=10),
            TaIndicatorSpec("dpo_20", "dpo", ("close",), {"length": 20, "lookahead": False}, min_rows=20),
            TaIndicatorSpec("kst", "kst", ("close",), min_rows=30),
            TaIndicatorSpec("stc", "stc", ("close",), min_rows=50),
            TaIndicatorSpec("tsi", "tsi", ("close",), min_rows=25),
            TaIndicatorSpec("trix", "trix", ("close",), min_rows=18),
            TaIndicatorSpec("uo", "uo", ("high", "low", "close"), min_rows=28),
            TaIndicatorSpec("ui_14", "ui", ("close",), {"length": 14}, min_rows=14),
            TaIndicatorSpec("natr_14", "natr", ("high", "low", "close"), {"length": 14}, min_rows=14),
            TaIndicatorSpec("pvo", "pvo", ("volume",), min_rows=26),
            TaIndicatorSpec("pvol", "pvol", ("close", "volume")),
        ),
        "technical_overlap": (
            TaIndicatorSpec("sma_10", "sma", ("close",), {"length": 10}, min_rows=10),
            TaIndicatorSpec("sma_20", "sma", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("sma_50", "sma", ("close",), {"length": 50}, min_rows=50),
            TaIndicatorSpec("ema_12", "ema", ("close",), {"length": 12}, min_rows=12),
            TaIndicatorSpec("ema_26", "ema", ("close",), {"length": 26}, min_rows=26),
            TaIndicatorSpec("ema_50", "ema", ("close",), {"length": 50}, min_rows=50),
            TaIndicatorSpec("hma_20", "hma", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("wma_20", "wma", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("kama_20", "kama", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("vwma_20", "vwma", ("close", "volume"), {"length": 20}, min_rows=20),
            TaIndicatorSpec("vwap", "vwap", ("high", "low", "close", "volume"), min_rows=2),
            TaIndicatorSpec("bbands_20", "bbands", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("kc_20", "kc", ("high", "low", "close"), {"length": 20}, min_rows=20),
            TaIndicatorSpec("donchian_20", "donchian", ("high", "low"), {"lower_length": 20, "upper_length": 20}, min_rows=20),
            TaIndicatorSpec("supertrend_7_3", "supertrend", ("high", "low", "close"), {"length": 7, "multiplier": 3.0}, min_rows=7),
            TaIndicatorSpec("psar", "psar", ("high", "low", "close")),
            TaIndicatorSpec("midpoint_20", "midpoint", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("midprice_20", "midprice", ("high", "low"), {"length": 20}, min_rows=20),
            TaIndicatorSpec("hl2", "hl2", ("high", "low")),
            TaIndicatorSpec("hlc3", "hlc3", ("high", "low", "close")),
            TaIndicatorSpec("ohlc4", "ohlc4", ("open", "high", "low", "close")),
        ),
        "technical_performance": (
            TaIndicatorSpec("pct_return_1", "percent_return", ("close",), {"length": 1}, min_rows=2),
            TaIndicatorSpec("pct_return_5", "percent_return", ("close",), {"length": 5}, min_rows=5),
            TaIndicatorSpec("pct_return_20", "percent_return", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("log_return_1", "log_return", ("close",), {"length": 1}, min_rows=2),
            TaIndicatorSpec("log_return_5", "log_return", ("close",), {"length": 5}, min_rows=5),
            TaIndicatorSpec("log_return_20", "log_return", ("close",), {"length": 20}, min_rows=20),
            TaIndicatorSpec("drawdown", "drawdown", ("close",)),
        ),
    }


def build_price_ta_classic_curated_feature_families(symbol: str, df_prices: pd.DataFrame) -> dict[str, object]:
    if df_prices.empty:
        return {family_name: _to_built_feature_set(symbol, pd.DataFrame(), []) for family_name in TA_CLASSIC_FAMILY_PREFIXES}
    ta = _import_pandas_ta_classic()
    prices = _prepare_price_frame(df_prices)
    if prices.empty:
        return {family_name: _to_built_feature_set(symbol, pd.DataFrame(), []) for family_name in TA_CLASSIC_FAMILY_PREFIXES}
    result = {}
    with _suppress_pandas_ta_classic_row_warnings():
        for family_name, specs in _curated_ta_indicator_specs(ta).items():
            columns = {}
            for spec in specs:
                indicator = _compute_indicator(ta, prices, spec)
                if indicator.empty:
                    continue
                for column in indicator.columns:
                    out_col = _feature_column_name(family_name, spec.name, column)
                    columns[out_col] = pd.to_numeric(indicator[column], errors="coerce")
            frame = pd.DataFrame(columns, index=prices.index) if columns else pd.DataFrame(index=prices.index)
            feature_cols = _usable_feature_cols(frame)
            result[family_name] = _to_built_feature_set(symbol, frame, feature_cols)
    return result


def _manifest_matches() -> bool:
    if REBUILD_FEATURE_FAMILY_CACHE or not FEATURE_FAMILY_MANIFEST_PATH.exists():
        return False
    try:
        manifest = json.loads(FEATURE_FAMILY_MANIFEST_PATH.read_text(encoding="utf-8"))
    except Exception:
        return False
    expected_symbols = list(map(str, screened_equity_symbols))
    expected_sources = [f"{source}.{family}" for source, family in REQUESTED_FAMILY_PAIRS]
    if manifest.get("min_market_cap") != int(MIN_MARKET_CAP):
        return False
    if manifest.get("symbols") != expected_symbols:
        return False
    if manifest.get("strategy_sources") != expected_sources:
        return False
    index_path = FEATURE_FAMILY_CACHE_DIR / "index.csv"
    if not index_path.exists():
        return False
    try:
        cached_index = pd.read_csv(index_path)
    except Exception:
        return False
    if "strategy_source" not in cached_index.columns or "features" not in cached_index.columns:
        return False
    feature_counts = dict(zip(cached_index["strategy_source"].astype(str), pd.to_numeric(cached_index["features"], errors="coerce").fillna(0).astype(int)))
    for source, family in REQUESTED_FAMILY_PAIRS:
        panel_path, metadata_path = _feature_family_paths(source, family)
        if not panel_path.exists() or not metadata_path.exists():
            return False
        if feature_counts.get(f"{source}.{family}", 0) <= 0:
            return False
    return True


def _write_feature_family_cache() -> None:
    started = perf_counter()
    family_frames: dict[tuple[str, str], list[pd.DataFrame]] = {pair: [] for pair in REQUESTED_FAMILY_PAIRS}
    family_specs: dict[tuple[str, str], list[object]] = {pair: [] for pair in REQUESTED_FAMILY_PAIRS}
    base_frames: list[pd.DataFrame] = []
    diagnostics: list[dict[str, object]] = []
    total_symbols = len(screened_equity_symbols)

    for i, raw_symbol in enumerate(screened_equity_symbols, start=1):
        symbol = str(raw_symbol).strip().upper()
        frame, specs, diag = _build_symbol_fundamental_panel(engineering_warehouse, symbol, feature_config)
        diagnostics.append(diag)
        if frame.empty:
            continue
        if i == 1 or i % 25 == 0 or i == total_symbols:
            print(f"[feature-cache] built symbol {i:,}/{total_symbols:,}: {symbol} rows={len(frame):,} features={len(specs):,}", flush=True)

        group_pe_source_cols = [f"fmp_daily_mcap_multiple__{metric}" for metric in GROUP_PE_FEATURES if f"fmp_daily_mcap_multiple__{metric}" in frame.columns]
        base = frame[["symbol", "date", "close", "daily_market_cap", *group_pe_source_cols]].copy()
        for numeric_col in ["close", "daily_market_cap", *group_pe_source_cols]:
            base[numeric_col] = pd.to_numeric(base[numeric_col], errors="coerce").astype("float32")
        base_frames.append(base)

        meta = _metadata_frame(specs)
        for source, family in PER_SYMBOL_FAMILIES:
            family_meta = meta.loc[meta["source"].eq(source) & meta["family"].eq(family)].copy()
            feature_cols = list(dict.fromkeys(col for col in family_meta["feature"].tolist() if col in frame.columns))
            if not feature_cols and (source, family) in {("fmp", "fmp_institutional_position_summary"), ("fmp", "fmp_quarterly_financial_estimates"), ("fmp", "fmp_historical_ratings"), ("fmp", "fmp_esg_scores"), ("fmp", "fmp_company_news")} and family_specs[(source, family)]:
                # Preserve the family panel for symbols whose provider
                # history is missing; their values remain NaN rather
                # than dropping the symbol from the fused universe.
                feature_cols = [spec.feature for spec in family_specs[(source, family)]]
                frame = frame[["symbol", "date"]].copy()
                for column in feature_cols:
                    frame[column] = np.nan
            if not feature_cols:
                continue
            family_frames[(source, family)].append(_downcast_numeric_columns(frame[["symbol", "date", *feature_cols]], feature_cols))
            family_specs[(source, family)].extend(spec for spec in specs if spec.source == source and spec.family == family)

        if REQUESTED_TECHNICAL_FAMILY_PAIRS:
            try:
                price_frame = _slice_frame(
                    engineering_warehouse.read_prices(symbol, provider=feature_config.provider),
                    feature_config.start_date,
                    feature_config.end_date,
                )
            except Exception as exc:
                diagnostics.append({"symbol": symbol, "status": "technical_price_read_failed", "error": str(exc)})
                price_frame = pd.DataFrame()
            if not price_frame.empty:
                requested_ta_families = {family for source, family in REQUESTED_TECHNICAL_FAMILY_PAIRS if source == "fmp" and family in TA_CLASSIC_FAMILY_PREFIXES}
                if requested_ta_families:
                    try:
                        ta_family_sets = build_price_ta_classic_curated_feature_families(symbol, price_frame)
                    except Exception as exc:
                        diagnostics.append({"symbol": symbol, "status": "ta_classic_technicals_failed", "error": str(exc)})
                        ta_family_sets = {}
                    for family in sorted(requested_ta_families):
                        feature_set = ta_family_sets.get(family)
                        if feature_set is None:
                            continue
                        tech_panel, tech_cols = _built_feature_set_to_panel(symbol, feature_set)
                        if tech_cols:
                            family_frames[("fmp", family)].append(tech_panel)
                            family_specs[("fmp", family)].extend(_technical_feature_specs(family, tech_cols))
        del frame, specs
        if i % 50 == 0:
            gc.collect()

    if not base_frames:
        raise RuntimeError("No base feature frames were built for the requested symbols.")

    base_panel = pd.concat(base_frames, ignore_index=True, sort=False).sort_values(["date", "symbol"]).reset_index(drop=True)
    del base_frames
    gc.collect()

    context_specs = []
    if REQUESTED_CONTEXT_FAMILIES:
        context_specs.extend(_add_time_calendar_features(base_panel))
        context_specs.extend(_add_macro_context_features(engineering_warehouse, base_panel, feature_config))
        context_specs.extend(_add_cross_symbol_context_features(engineering_warehouse, base_panel, feature_config))
        context_meta = _metadata_frame(context_specs)
        for source, family in REQUESTED_CONTEXT_FAMILIES:
            family_meta = context_meta.loc[context_meta["source"].eq(source) & context_meta["family"].eq(family)].copy()
            feature_cols = [col for col in family_meta["feature"].tolist() if col in base_panel.columns]
            if not feature_cols:
                continue
            family_frames[(source, family)].append(_downcast_numeric_columns(base_panel[["symbol", "date", *feature_cols]], feature_cols))
            family_specs[(source, family)].extend(spec for spec in context_specs if spec.source == source and spec.family == family)
    del base_panel
    gc.collect()

    index_rows = []
    all_metadata = []
    for source, family in REQUESTED_FAMILY_PAIRS:
        panel_path, metadata_path = _feature_family_paths(source, family)
        parts = family_frames.get((source, family), [])
        metadata = _metadata_frame(family_specs.get((source, family), []))
        if parts and not metadata.empty:
            panel = pd.concat(parts, ignore_index=True, sort=False).sort_values(["date", "symbol"]).reset_index(drop=True)
            feature_cols = [col for col in metadata["feature"].tolist() if col in panel.columns]
            panel = panel[["symbol", "date", *feature_cols]].copy()
            panel.to_parquet(panel_path, index=False)
            metadata.to_parquet(metadata_path, index=False)
            rows = len(panel)
            memory_mb = float(panel.memory_usage(deep=True).sum() / 1024 / 1024)
            all_metadata.append(metadata)
            del panel
        else:
            pd.DataFrame(columns=["symbol", "date"]).to_parquet(panel_path, index=False)
            metadata.to_parquet(metadata_path, index=False)
            rows = 0
            memory_mb = 0.0
            feature_cols = []
        index_rows.append(
            {
                "source": source,
                "family": family,
                "strategy_source": f"{source}.{family}",
                "panel_path": str(panel_path),
                "metadata_path": str(metadata_path),
                "rows": rows,
                "features": len(feature_cols),
                "memory_mb_before_write": memory_mb,
            }
        )
        family_frames[(source, family)] = []
        gc.collect()

    manifest = {
        "min_market_cap": int(MIN_MARKET_CAP),
        "start_date": DATA_START,
        "end_date": optional_date(DATA_END),
        "symbols": list(map(str, screened_equity_symbols)),
        "strategy_sources": [f"{source}.{family}" for source, family in REQUESTED_FAMILY_PAIRS],
        "elapsed_seconds": perf_counter() - started,
    }
    FEATURE_FAMILY_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
    pd.DataFrame(index_rows).to_csv(FEATURE_FAMILY_CACHE_DIR / "index.csv", index=False)
    pd.DataFrame(diagnostics).to_csv(FEATURE_FAMILY_CACHE_DIR / "symbol_diagnostics.csv", index=False)


if not _manifest_matches():
    print(f"Building feature-family cache at {FEATURE_FAMILY_CACHE_DIR}")
    _write_feature_family_cache()
else:
    print(f"Using cached feature-family panels from {FEATURE_FAMILY_CACHE_DIR}")

feature_family_index = pd.read_csv(FEATURE_FAMILY_CACHE_DIR / "index.csv")
metadata_frames = []
for row in feature_family_index.to_dict("records"):
    metadata_path = Path(row["metadata_path"])
    if metadata_path.exists():
        metadata = pd.read_parquet(metadata_path)
        if not metadata.empty:
            metadata_frames.append(metadata)
selected_feature_metadata = (
    pd.concat(metadata_frames, ignore_index=True, sort=False).drop_duplicates().reset_index(drop=True)
    if metadata_frames
    else pd.DataFrame(columns=["feature", "family", "source", "source_column", "expected_direction"])
)

if RUN_REGIME_YEAR_FEATURE:
    regime_metadata = feature_family_index.loc[feature_family_index["features"].gt(0), ["source", "family"]].copy()
    regime_metadata["feature"] = REGIME_YEAR_FEATURE
    regime_metadata["source_column"] = "date.year"
    regime_metadata["expected_direction"] = "neutral"
    regime_metadata = regime_metadata[["feature", "family", "source", "source_column", "expected_direction"]]
    selected_feature_metadata = (
        pd.concat([selected_feature_metadata, regime_metadata], ignore_index=True, sort=False)
        .drop_duplicates(subset=["source", "family", "feature"])
        .reset_index(drop=True)
    )

available_sources = set(feature_family_index.loc[feature_family_index["features"].gt(0), "strategy_source"].astype(str))
wanted_sources = {str(source).strip() for source in EXPERIMENT_STRATEGY_SOURCES}
missing_requested_sources = sorted(wanted_sources.difference(available_sources))
if REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES and missing_requested_sources:
    raise RuntimeError(f"Requested feature families are missing from feature cache: {missing_requested_sources}")

print(
    {
        "symbols": len(screened_equity_symbols),
        "feature_family_cache_dir": str(FEATURE_FAMILY_CACHE_DIR),
        "feature_families": len(feature_family_index),
        "feature_rows_total_across_family_panels": int(feature_family_index["rows"].sum()),
        "feature_columns_total_across_families": int(feature_family_index["features"].sum()),
        "missing_requested_families": missing_requested_sources,
        "regime_year_feature_added_to_each_family": bool(RUN_REGIME_YEAR_FEATURE),
        "technical_feature_families_requested": [f"{source}.{family}" for source, family in REQUESTED_TECHNICAL_FAMILY_PAIRS],
    }
)
display(feature_family_index.sort_values(["source", "family"]).reset_index(drop=True))
display(
    selected_feature_metadata.groupby(["source", "family"], as_index=False)
    .agg(feature_count=("feature", "nunique"))
    .sort_values(["source", "family"])
)


## Experiment Plan

Scale assumptions for this notebook:

- The default run is the 10B market-cap equity universe. The universe size is controlled only by `MIN_MARKET_CAP`.
- FMP refresh is off by default; the experiment uses cached warehouse data unless `RUN_FMP_REFRESH=True` is set deliberately.
- Base feature-family classifiers train on pre-2021 rows, with diagnostics and autoencoders disabled.
- Meta-model training scores only oracle-label dates to avoid full daily scoring during training.
- Post-2021 backtesting still scores every available day, but writes reusable yearly chunks under a run-specific artifact directory.
- Symbol-level `backtesting.py` checks are checkpointed and resumable so a crash does not restart every symbol/strategy pair.


In [ ]:
from quant_orchestrator.platforms.ml_frameworks.rapids.random_forest import rapids_gpu_info

feature_family_plan = pd.DataFrame(
    [
        {"strategy_source": name, "provider": name.split(".", 1)[0], "feature_family": name.split(".", 1)[1]}
        for name in EXPERIMENT_STRATEGY_SOURCES
    ]
)
classifier_plan = feature_family_plan.assign(
    base_classifier="RapidsRandomForestClassifier",
    base_train_window=f"<= {BASE_TRAIN_END}",
    meta_feature="family long/short/net score",
    meta_classifier="RapidsRandomForestClassifier",
    regime_year_feature=REGIME_YEAR_FEATURE if RUN_REGIME_YEAR_FEATURE else None,
    meta_train_window=f"<= {BASE_TRAIN_END}",
    oos_window=f">= {META_OOS_START}",
    trading_rule=f"long_score > {SCORE_THRESHOLD}; sell when < {SCORE_THRESHOLD}",
)

print("Feature-family classifier and meta-model plan")
display(classifier_plan)
print("RAPIDS GPU preflight")
display(rapids_gpu_info())


## Train Base Feature-Family Models


In [ ]:
import gc
from quant_orchestrator.platforms.ml_frameworks.rapids.random_forest import RapidsRandomForestClassifier
from quant_orchestrator.research_tools.ml_trading import (
    build_family_prediction_frame,
    build_strategy_score_frame,
    classification_probability_diagnostics,
    prepare_family_dataset,
)
from quant_orchestrator.research_tools.ml_trading_experiment import MLTradingExperimentConfig

required_training_inputs = ["screened_equity_symbols", "feature_family_index", "selected_feature_metadata", "oracle_label_rows"]
missing_training_inputs = [name for name in required_training_inputs if name not in globals()]
if missing_training_inputs:
    raise RuntimeError(f"Run refresh, target, and feature-family cache cells first. Missing: {missing_training_inputs}")

base_config = MLTradingExperimentConfig(
    experiment_name="trading_app_v2_equity_meta_base_10b",
    mode="classifier",
    min_market_cap=MIN_MARKET_CAP,
    symbols=tuple(screened_equity_symbols),
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
    train_end=BASE_TRAIN_END,
    oos_start=META_OOS_START,
    score_start=DATA_START,
    fit_all_available_data=False,
    refresh_missing_fmp_data=False,
    top_k_values=(1,),
    entry_threshold=SCORE_THRESHOLD,
    exit_threshold=SCORE_THRESHOLD,
    strategy_sources=tuple(EXPERIMENT_STRATEGY_SOURCES),
    feature_representations=("raw",),
    ae_config=None,
    target_label_mode="oracle_only",
    oracle_frequencies=("YE",),
    oracle_k_min=1,
    oracle_k_max=12,
    rf_params=BASE_RF_PARAMS,
    quant_warehouse_root=None,
    run_zipline_backtests=False,
    include_yearly_vectorized_diagnostics=False,
    backtesting_py_symbol_cases_per_side=0,
    run_model_diagnostics=False,
    run_trading_diagnostics=False,
    log_mlflow=False,
    random_seed=RANDOM_SEED,
)

display(asdict(base_config))


def load_feature_family_panel(source: str, family: str) -> pd.DataFrame:
    match = feature_family_index.loc[
        feature_family_index["source"].astype(str).eq(str(source))
        & feature_family_index["family"].astype(str).eq(str(family))
    ]
    if match.empty:
        return pd.DataFrame(columns=["symbol", "date"])
    path = Path(match.iloc[0]["panel_path"])
    if not path.exists():
        return pd.DataFrame(columns=["symbol", "date"])
    panel = pd.read_parquet(path)
    if panel.empty:
        return panel
    panel["symbol"] = panel["symbol"].astype(str).str.upper()
    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize()
    if RUN_REGIME_YEAR_FEATURE and REGIME_YEAR_FEATURE not in panel.columns:
        panel[REGIME_YEAR_FEATURE] = panel["date"].dt.year.astype("float32")
    return panel


def add_family_score_rank_columns(strategy_scores: pd.DataFrame) -> pd.DataFrame:
    if strategy_scores.empty:
        return strategy_scores.copy()
    out = strategy_scores.copy()
    out["symbol"] = out["symbol"].astype(str).str.upper()
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize()
    out["long_score"] = pd.to_numeric(out["long_score"], errors="coerce")
    out["short_score"] = pd.to_numeric(out["short_score"], errors="coerce")
    out["net_score"] = out["long_score"] - out["short_score"]
    group_keys = ["strategy_source", "date"]
    out["long_score_rank"] = out.groupby(group_keys)["long_score"].rank(method="average", pct=True)
    out["short_score_rank"] = out.groupby(group_keys)["short_score"].rank(method="average", pct=True)
    out["net_score_rank"] = out.groupby(group_keys)["net_score"].rank(method="average", pct=True)
    for col in ["long_score_rank", "short_score_rank", "net_score_rank"]:
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float32")
    return out


def _base_ensemble_mean_scores(single_model_scores: pd.DataFrame) -> pd.DataFrame:
    mean_scores = (
        single_model_scores.groupby(["symbol", "date"], as_index=False)
        .agg(
            long_score=("long_score", "mean"),
            short_score=("short_score", "mean"),
            long_exit_score=("long_exit_score", "mean"),
            short_exit_score=("short_exit_score", "mean"),
            classifier_long_score=("classifier_long_score", "mean"),
            classifier_short_score=("classifier_short_score", "mean"),
            long_agree_count=("long_agree_count", "sum"),
            short_agree_count=("short_agree_count", "sum"),
            ae_familiarity=("ae_familiarity", "mean"),
            ae_recon_error=("ae_recon_error", "mean"),
            ae_latent_distance=("ae_latent_distance", "mean"),
            model_count=("strategy_source", "nunique"),
        )
    )
    mean_scores["source"] = "ensemble"
    mean_scores["family"] = "mean"
    mean_scores["strategy_source"] = "ensemble_mean"
    mean_scores["net_score"] = mean_scores["long_score"] - mean_scores["short_score"]
    return mean_scores


def _rank_mean_ensemble_scores(ranked_single_model_scores: pd.DataFrame) -> pd.DataFrame:
    rank_scores = (
        ranked_single_model_scores.groupby(["symbol", "date"], as_index=False)
        .agg(
            long_score=("long_score_rank", "mean"),
            short_score=("short_score_rank", "mean"),
            long_exit_score=("long_score_rank", "mean"),
            short_exit_score=("short_score_rank", "mean"),
            classifier_long_score=("long_score_rank", "mean"),
            classifier_short_score=("short_score_rank", "mean"),
            long_agree_count=("long_score", lambda values: int(pd.to_numeric(values, errors="coerce").gt(SCORE_THRESHOLD).sum())),
            short_agree_count=("short_score", lambda values: int(pd.to_numeric(values, errors="coerce").gt(SCORE_THRESHOLD).sum())),
            model_count=("strategy_source", "nunique"),
        )
    )
    rank_scores["ae_familiarity"] = np.nan
    rank_scores["ae_recon_error"] = np.nan
    rank_scores["ae_latent_distance"] = np.nan
    rank_scores["source"] = "ensemble"
    rank_scores["family"] = "rank_mean"
    rank_scores["strategy_source"] = "ensemble_rank_mean"
    rank_scores["net_score"] = rank_scores["long_score"] - rank_scores["short_score"]
    return rank_scores


def build_ensemble_mean_scores(single_model_scores: pd.DataFrame) -> pd.DataFrame:
    if single_model_scores.empty:
        return pd.DataFrame()
    ranked_scores = add_family_score_rank_columns(single_model_scores)
    parts = [_base_ensemble_mean_scores(ranked_scores)]
    if INCLUDE_RANK_MEAN_ENSEMBLE:
        parts.append(_rank_mean_ensemble_scores(ranked_scores))
    return pd.concat(parts, ignore_index=True, sort=False)


def extract_classifier_feature_importances(classifier: RapidsRandomForestClassifier, features: list[str]) -> pd.Series:
    values = getattr(classifier.model, "feature_importances_", None)
    if values is None:
        return pd.Series(dtype="float64")
    try:
        if hasattr(values, "to_numpy"):
            values = values.to_numpy()
        elif hasattr(values, "get"):
            values = values.get()
        values = np.asarray(values, dtype="float64").reshape(-1)
    except Exception:
        return pd.Series(dtype="float64")
    if len(values) != len(features):
        return pd.Series(dtype="float64")
    return pd.Series(values, index=list(features), dtype="float64")


def regime_year_feature_importance(classifier: RapidsRandomForestClassifier, features: list[str]) -> float:
    if not RUN_REGIME_YEAR_FEATURE or REGIME_YEAR_FEATURE not in features:
        return np.nan
    importances = extract_classifier_feature_importances(classifier, features)
    return float(importances.get(REGIME_YEAR_FEATURE, np.nan)) if not importances.empty else np.nan


def score_trained_family_panel(
    panel: pd.DataFrame,
    payload: dict,
    *,
    source: str,
    family: str,
    score_start: pd.Timestamp,
    score_keys: pd.DataFrame | None = None,
) -> pd.DataFrame:
    if panel.empty:
        return pd.DataFrame()
    work = panel
    if score_keys is not None:
        work = work.merge(score_keys, on=["symbol", "date"], how="inner")
    if work.empty:
        return pd.DataFrame()
    pred_input = build_family_prediction_frame(
        work,
        payload["raw_features"],
        min_feature_coverage=base_config.min_feature_coverage,
    )
    if pred_input.empty:
        return pd.DataFrame()
    pred_input = pred_input.loc[pd.to_datetime(pred_input["date"], errors="coerce").ge(score_start)].copy()
    if pred_input.empty:
        return pd.DataFrame()
    proba = payload["classifier"].predict_proba_frame(pred_input, payload["features"])
    return build_strategy_score_frame(
        source=str(source),
        family=str(family),
        prediction_frame=pred_input,
        probability_frame=proba,
        apply_ae_to_exits=False,
    )


FAMILY_SCORE_CACHE_DIR = RUN_ARTIFACT_DIR / "family_score_cache"
FAMILY_SCORE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
FAMILY_SCORE_CACHE_VERSION = 1


def _jsonable_for_cache(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): _jsonable_for_cache(v) for k, v in sorted(value.items(), key=lambda item: str(item[0]))}
    if isinstance(value, (list, tuple, set)):
        return [_jsonable_for_cache(v) for v in value]
    return value


def _stable_cache_payload(value) -> str:
    return json.dumps(_jsonable_for_cache(value), sort_keys=True, separators=(",", ":"), default=str)


def _stable_cache_hash(value) -> str:
    import hashlib

    return hashlib.sha256(_stable_cache_payload(value).encode("utf-8")).hexdigest()


def _read_json_cache_payload(path: Path) -> dict:
    try:
        return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    except Exception:
        return {}


def _feature_panel_file_fingerprints() -> list[dict]:
    rows = []
    for row in feature_family_index.sort_values(["source", "family"]).to_dict("records"):
        panel_path = Path(row.get("panel_path", ""))
        metadata_path = Path(row.get("metadata_path", ""))
        panel_stat = panel_path.stat() if panel_path.exists() else None
        metadata_stat = metadata_path.stat() if metadata_path.exists() else None
        rows.append(
            {
                "strategy_source": str(row.get("strategy_source")),
                "rows": int(row.get("rows") or 0),
                "features": int(row.get("features") or 0),
                "panel_path": str(panel_path),
                "panel_size": int(panel_stat.st_size) if panel_stat else None,
                "panel_mtime_ns": int(panel_stat.st_mtime_ns) if panel_stat else None,
                "metadata_path": str(metadata_path),
                "metadata_size": int(metadata_stat.st_size) if metadata_stat else None,
                "metadata_mtime_ns": int(metadata_stat.st_mtime_ns) if metadata_stat else None,
            }
        )
    return rows


def _family_score_cache_manifest(
    *,
    cache_name: str,
    score_scope: str,
    train_end: str,
    score_start: str,
    score_end: str | None = None,
    score_year: int | None = None,
    score_key_rows: int | None = None,
    score_key_hash: str | None = None,
) -> dict:
    feature_manifest = _read_json_cache_payload(FEATURE_FAMILY_MANIFEST_PATH)
    panel_fingerprints = _feature_panel_file_fingerprints()
    return {
        "cache_version": FAMILY_SCORE_CACHE_VERSION,
        "cache_name": cache_name,
        "score_scope": score_scope,
        "score_year": score_year,
        "score_start": score_start,
        "score_end": score_end,
        "score_key_rows": score_key_rows,
        "score_key_hash": score_key_hash,
        "min_market_cap": int(MIN_MARKET_CAP),
        "symbols": list(map(str, screened_equity_symbols)),
        "data_start": DATA_START,
        "data_end": optional_date(DATA_END),
        "train_end": train_end,
        "meta_oos_start": META_OOS_START,
        "random_seed": int(RANDOM_SEED),
        "base_rf_params": _jsonable_for_cache(BASE_RF_PARAMS),
        "strategy_sources": list(map(str, EXPERIMENT_STRATEGY_SOURCES)),
        "run_regime_year_feature": bool(RUN_REGIME_YEAR_FEATURE),
        "regime_year_feature": REGIME_YEAR_FEATURE,
        "include_technical_feature_families": bool(INCLUDE_TECHNICAL_FEATURE_FAMILIES),
        "feature_family_manifest_hash": _stable_cache_hash(feature_manifest),
        "feature_panel_fingerprint_hash": _stable_cache_hash(panel_fingerprints),
        "target_label_manifest_hash": _stable_cache_hash(_read_json_cache_payload(ORACLE_LABEL_MANIFEST_PATH)),
        "target_label_rows_fingerprint_hash": _stable_cache_hash(
            {
                "path": str(ORACLE_LABEL_ROWS_PATH),
                "size": ORACLE_LABEL_ROWS_PATH.stat().st_size if ORACLE_LABEL_ROWS_PATH.exists() else None,
                "mtime_ns": ORACLE_LABEL_ROWS_PATH.stat().st_mtime_ns if ORACLE_LABEL_ROWS_PATH.exists() else None,
            }
        ),
    }


def _family_score_cache_paths(cache_name: str) -> tuple[Path, Path]:
    return FAMILY_SCORE_CACHE_DIR / f"{cache_name}.parquet", FAMILY_SCORE_CACHE_DIR / f"{cache_name}.manifest.json"


def _normalize_family_score_cache_frame(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    keep = [
        "strategy_source",
        "source",
        "family",
        "symbol",
        "date",
        "long_score",
        "short_score",
        "long_exit_score",
        "short_exit_score",
        "net_score",
        "model_count",
        "long_score_rank",
        "short_score_rank",
        "net_score_rank",
    ]
    out = frame[[col for col in keep if col in frame.columns]].copy()
    out["symbol"] = out["symbol"].astype(str).str.upper()
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize()
    if "net_score" not in out.columns and {"long_score", "short_score"}.issubset(out.columns):
        out["net_score"] = pd.to_numeric(out["long_score"], errors="coerce") - pd.to_numeric(out["short_score"], errors="coerce")
    for col in ["long_score", "short_score", "long_exit_score", "short_exit_score", "net_score", "model_count", "long_score_rank", "short_score_rank", "net_score_rank"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("float32")
    return out.sort_values(["date", "strategy_source", "symbol"]).reset_index(drop=True)


def _family_score_cache_is_valid(cache_name: str, expected_manifest: dict) -> bool:
    if not REUSE_FAMILY_SCORE_CACHE or REBUILD_FAMILY_SCORE_CACHE:
        return False
    data_path, manifest_path = _family_score_cache_paths(cache_name)
    if not data_path.exists() or not manifest_path.exists():
        return False
    cached_manifest = _read_json_cache_payload(manifest_path)
    return _stable_cache_hash(cached_manifest) == _stable_cache_hash(expected_manifest)


def _load_family_score_cache(cache_name: str, expected_manifest: dict) -> pd.DataFrame | None:
    if not _family_score_cache_is_valid(cache_name, expected_manifest):
        return None
    data_path, _manifest_path = _family_score_cache_paths(cache_name)
    scores = pd.read_parquet(data_path)
    scores = _normalize_family_score_cache_frame(scores)
    print(f"[family-score-cache] loaded {cache_name}: rows={len(scores):,} path={data_path}")
    return scores


def _write_family_score_cache(cache_name: str, scores: pd.DataFrame, manifest: dict) -> None:
    if scores.empty:
        return
    data_path, manifest_path = _family_score_cache_paths(cache_name)
    data_path.parent.mkdir(parents=True, exist_ok=True)
    normalized = _normalize_family_score_cache_frame(scores)
    normalized.to_parquet(data_path, index=False)
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True, default=str), encoding="utf-8")
    print(f"[family-score-cache] wrote {cache_name}: rows={len(normalized):,} path={data_path}")


oracle_score_keys = oracle_label_rows[["symbol", "date"]].drop_duplicates().copy()
oracle_score_keys["symbol"] = oracle_score_keys["symbol"].astype(str).str.upper()
oracle_score_keys["date"] = pd.to_datetime(oracle_score_keys["date"], errors="coerce").dt.normalize()
oracle_score_key_hash = _stable_cache_hash(
    oracle_score_keys.sort_values(["date", "symbol"]).astype({"symbol": "string"}).assign(
        date=lambda frame: frame["date"].dt.strftime("%Y-%m-%d")
    ).to_dict("records")
)
base_oracle_score_cache_name = "base_oracle_family_scores"
base_oracle_score_manifest = _family_score_cache_manifest(
    cache_name=base_oracle_score_cache_name,
    score_scope="oracle_keys",
    train_end=BASE_TRAIN_END,
    score_start=DATA_START,
    score_end=optional_date(DATA_END),
    score_key_rows=len(oracle_score_keys),
    score_key_hash=oracle_score_key_hash,
)
use_cached_base_oracle_scores = _family_score_cache_is_valid(base_oracle_score_cache_name, base_oracle_score_manifest)

base_started = perf_counter()
model_rows = []
trained_models = {}
meta_score_frames = []
training_plan = feature_family_index.loc[feature_family_index["features"].gt(0)].sort_values(["source", "family"])
print(f"Streaming classifier training for {len(training_plan):,} feature families")
if use_cached_base_oracle_scores:
    print("[family-score-cache] reusing cached oracle-key family scores for meta training; base models still train for daily scoring.")

for job_index, row in enumerate(training_plan.to_dict("records"), start=1):
    source = str(row["source"])
    family = str(row["family"])
    strategy_source = f"{source}.{family}"
    panel = load_feature_family_panel(source, family)
    metadata = selected_feature_metadata.loc[
        selected_feature_metadata["source"].astype(str).eq(source)
        & selected_feature_metadata["family"].astype(str).eq(family)
    ].copy()
    family_frame, features = prepare_family_dataset(
        panel,
        metadata,
        oracle_label_rows,
        source=source,
        family=family,
        min_feature_coverage=base_config.min_feature_coverage,
    )
    train = family_frame.loc[pd.to_datetime(family_frame["date"], errors="coerce").le(pd.Timestamp(BASE_TRAIN_END))].copy()
    oos = family_frame.loc[pd.to_datetime(family_frame["date"], errors="coerce").ge(pd.Timestamp(META_OOS_START))].copy()
    print(
        f"[base-train] {job_index}/{len(training_plan)} {strategy_source} "
        f"panel_rows={len(panel):,} train_rows={len(train):,} features={len(features):,}",
        flush=True,
    )
    if len(train) < base_config.min_train_rows_per_family or train["collapsed_label"].nunique() < base_config.min_classes_per_family:
        model_rows.append(
            {
                "source": source,
                "family": family,
                "strategy_source": strategy_source,
                "status": "skipped_sparse_train",
                "features": len(features),
                "rows": len(family_frame),
                "train_rows": len(train),
                "oos_rows": len(oos),
                "train_classes": train["collapsed_label"].nunique(),
                "regime_year_feature_included": bool(RUN_REGIME_YEAR_FEATURE and REGIME_YEAR_FEATURE in features),
                "regime_year_feature_importance": np.nan,
            }
        )
        del panel, metadata, family_frame, train, oos
        gc.collect()
        continue
    fit_started = perf_counter()
    classifier = RapidsRandomForestClassifier.fit(
        train,
        features=features,
        target_col="collapsed_label",
        random_state=RANDOM_SEED,
        params=BASE_RF_PARAMS,
    )
    classifier_fit_seconds = perf_counter() - fit_started
    payload = {
        "classifier": classifier,
        "autoencoder": None,
        "feature_autoencoder": None,
        "features": features,
        "raw_features": features,
        "base_family": family,
        "representation": "raw",
    }
    trained_models[(source, family)] = payload
    train_scores = {
        "rows": len(train),
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "macro_f1": np.nan,
        "log_loss": np.nan,
        "brier_macro": np.nan,
        "expected_calibration_error": np.nan,
        "mean_confidence": np.nan,
    }
    oos_scores = {
        "rows": len(oos),
        "accuracy": np.nan,
        "balanced_accuracy": np.nan,
        "macro_f1": np.nan,
        "log_loss": np.nan,
        "brier_macro": np.nan,
        "expected_calibration_error": np.nan,
        "mean_confidence": np.nan,
    }
    model_rows.append(
        {
            "source": source,
            "family": family,
            "base_family": family,
            "representation": "raw",
            "strategy_source": strategy_source,
            "status": "ok",
            "features": len(features),
            "raw_features": len(features),
            "ae_features": 0,
            "rows": len(family_frame),
            "train_rows": len(train),
            "oos_rows": len(oos),
            "training_window": "split",
            "classes": family_frame["collapsed_label"].nunique(),
            "regime_year_feature_included": bool(RUN_REGIME_YEAR_FEATURE and REGIME_YEAR_FEATURE in features),
            "regime_year_feature_importance": regime_year_feature_importance(classifier, features),
            "classifier_fit_seconds": classifier_fit_seconds,
            "classifier_backend": classifier.gpu_info.get("backend", "rapids_cuml_gpu"),
            "gpu_device_id": classifier.gpu_info.get("device_id"),
            "gpu_device_name": classifier.gpu_info.get("device_name"),
            "cudf_version": classifier.gpu_info.get("cudf_version"),
            "cuml_version": classifier.gpu_info.get("cuml_version"),
            "cupy_version": classifier.gpu_info.get("cupy_version"),
            "ae_fit_seconds": np.nan,
            **{f"train_{key}": value for key, value in train_scores.items()},
            **{f"oos_{key}": value for key, value in oos_scores.items()},
        }
    )
    if not use_cached_base_oracle_scores:
        scores = score_trained_family_panel(
            panel,
            payload,
            source=source,
            family=family,
            score_start=pd.Timestamp(DATA_START),
            score_keys=oracle_score_keys,
        )
        if not scores.empty:
            meta_score_frames.append(scores)
        del scores
    del panel, metadata, family_frame, train, oos
    gc.collect()

if use_cached_base_oracle_scores:
    base_single_scores_for_meta = _load_family_score_cache(base_oracle_score_cache_name, base_oracle_score_manifest)
    if base_single_scores_for_meta is None:
        base_single_scores_for_meta = pd.DataFrame()
else:
    base_single_scores_for_meta = pd.concat(meta_score_frames, ignore_index=True, sort=False) if meta_score_frames else pd.DataFrame()
    _write_family_score_cache(base_oracle_score_cache_name, base_single_scores_for_meta, base_oracle_score_manifest)
base_mean_scores_for_meta = build_ensemble_mean_scores(base_single_scores_for_meta)
base_strategy_scores_for_meta = pd.concat(
    [base_mean_scores_for_meta, base_single_scores_for_meta],
    ignore_index=True,
    sort=False,
) if not base_single_scores_for_meta.empty else pd.DataFrame()
base_model_results = pd.DataFrame(model_rows).reset_index(drop=True)

print({
    "elapsed_seconds": round(perf_counter() - base_started, 2),
    "trained_base_models": int((base_model_results["status"] == "ok").sum()) if "status" in base_model_results else len(base_model_results),
    "meta_training_score_rows": len(base_strategy_scores_for_meta),
    "score_sources": base_strategy_scores_for_meta["strategy_source"].nunique() if not base_strategy_scores_for_meta.empty else 0,
})
display(base_model_results.head(50))
regime_year_importance_summary = (
    base_model_results.loc[
        base_model_results["status"].eq("ok") & base_model_results["regime_year_feature_included"].fillna(False),
        ["strategy_source", "features", "train_rows", "regime_year_feature_importance"],
    ]
    .sort_values("regime_year_feature_importance", ascending=False, na_position="last")
    .reset_index(drop=True)
) if RUN_REGIME_YEAR_FEATURE and "regime_year_feature_importance" in base_model_results.columns else pd.DataFrame()
print("Regime-year feature importance by family; higher values indicate more regime/year sensitivity.")
display(regime_year_importance_summary)
display(base_strategy_scores_for_meta.head())


## Build Meta-Model Training Matrix


In [ ]:
def _safe_feature_name(text: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in str(text)).strip("_")


def build_meta_feature_panel(strategy_scores: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    family_scores = strategy_scores.loc[~strategy_scores["strategy_source"].isin(ENSEMBLE_STRATEGY_SOURCES)].copy()
    if family_scores.empty:
        return pd.DataFrame(columns=["symbol", "date"]), []
    family_scores = add_family_score_rank_columns(family_scores)
    frames = []
    feature_cols = []
    value_cols = ["long_score", "short_score", "net_score"]
    if INCLUDE_FAMILY_SCORE_RANK_FEATURES:
        value_cols.extend(["long_score_rank", "short_score_rank", "net_score_rank"])
    for value_col in value_cols:
        wide = family_scores.pivot_table(
            index=["symbol", "date"],
            columns="strategy_source",
            values=value_col,
            aggfunc="mean",
        )
        wide.columns = [f"{value_col}__{_safe_feature_name(col)}" for col in wide.columns]
        feature_cols.extend(wide.columns.tolist())
        frames.append(wide)
    panel = pd.concat(frames, axis=1).reset_index()
    numeric = panel[feature_cols].apply(pd.to_numeric, errors="coerce")
    medians = numeric.median(axis=0).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    panel[feature_cols] = numeric.replace([np.inf, -np.inf], np.nan).fillna(medians).astype("float32")
    return panel.sort_values(["date", "symbol"]).reset_index(drop=True), feature_cols


meta_feature_panel_train, meta_feature_cols = build_meta_feature_panel(base_strategy_scores_for_meta)
meta_dataset = oracle_label_rows.merge(meta_feature_panel_train, on=["symbol", "date"], how="inner")
meta_dataset["date"] = pd.to_datetime(meta_dataset["date"], errors="coerce").dt.normalize()
meta_train = meta_dataset.loc[meta_dataset["date"].le(pd.Timestamp(BASE_TRAIN_END))].copy()
meta_oos_labeled = meta_dataset.loc[meta_dataset["date"].ge(pd.Timestamp(META_OOS_START))].copy()

print({
    "meta_feature_rows_scored_on_oracle_dates": len(meta_feature_panel_train),
    "meta_feature_cols": len(meta_feature_cols),
    "meta_label_rows": len(meta_dataset),
    "meta_train_rows": len(meta_train),
    "meta_oos_labeled_rows": len(meta_oos_labeled),
    "meta_train_classes": sorted(meta_train["collapsed_label"].dropna().unique()) if not meta_train.empty else [],
    "meta_oos_classes": sorted(meta_oos_labeled["collapsed_label"].dropna().unique()) if not meta_oos_labeled.empty else [],
})
display(pd.DataFrame({"meta_feature": meta_feature_cols}).head(100))
display(meta_train.head())


## Train And Score Meta Classifier


In [ ]:
from quant_orchestrator.platforms.ml_frameworks.rapids.random_forest import RapidsRandomForestClassifier, ensure_probability_columns
from quant_orchestrator.research_tools.ml_trading import build_strategy_score_frame, classification_probability_diagnostics

if len(meta_train) < 250 or meta_train["collapsed_label"].nunique() < 2:
    raise RuntimeError(
        "Not enough pre-2021 meta training rows/classes. "
        f"rows={len(meta_train)}, classes={meta_train['collapsed_label'].nunique()}"
    )

meta_started = perf_counter()
meta_classifier = RapidsRandomForestClassifier.fit(
    meta_train,
    features=meta_feature_cols,
    target_col="collapsed_label",
    random_state=RANDOM_SEED,
    params=META_RF_PARAMS,
)

meta_train_proba = meta_classifier.predict_proba_frame(meta_train, meta_feature_cols)
meta_oos_labeled_proba = meta_classifier.predict_proba_frame(meta_oos_labeled, meta_feature_cols) if not meta_oos_labeled.empty else pd.DataFrame()
meta_train_diag = classification_probability_diagnostics(
    meta_train,
    meta_train_proba,
    target_col="collapsed_label",
    labels=meta_classifier.encoder.classes_,
)
meta_oos_diag = classification_probability_diagnostics(
    meta_oos_labeled,
    meta_oos_labeled_proba,
    target_col="collapsed_label",
    labels=meta_classifier.encoder.classes_,
) if not meta_oos_labeled.empty else {"rows": 0}

meta_model_results = pd.DataFrame([
    {
        "strategy_source": "stacked_meta",
        "source": "stacked",
        "family": "meta",
        "status": "ok",
        "features": len(meta_feature_cols),
        "train_rows": len(meta_train),
        "oos_labeled_rows": len(meta_oos_labeled),
        "training_window": f"<= {BASE_TRAIN_END}",
        "classifier_backend": meta_classifier.gpu_info.get("backend", "rapids_cuml_gpu"),
        "classifier_fit_seconds": perf_counter() - meta_started,
        **{f"train_{key}": value for key, value in meta_train_diag.items()},
        **{f"oos_{key}": value for key, value in meta_oos_diag.items()},
    }
])

display(meta_model_results)


## Secondary ML Diagnostics

In [ ]:
def _diagnose_score_frame_against_labels(score_frame: pd.DataFrame, labels: pd.DataFrame, *, name: str) -> dict[str, object]:
    merged = labels.merge(
        score_frame[["symbol", "date", "long_score", "short_score"]],
        on=["symbol", "date"],
        how="inner",
    )
    if merged.empty:
        return {"strategy_source": name, "rows": 0}
    proba = pd.DataFrame(
        {
            "prob__oracle_long": pd.to_numeric(merged["long_score"], errors="coerce").fillna(0.0),
            "prob__oracle_short": pd.to_numeric(merged["short_score"], errors="coerce").fillna(0.0),
        },
        index=merged.index,
    )
    diag = classification_probability_diagnostics(
        merged,
        proba,
        target_col="collapsed_label",
        labels=["oracle_long", "oracle_short"],
    )
    diag["strategy_source"] = name
    return diag

base_ensemble_labeled_oos = base_strategy_scores_for_meta.loc[
    base_strategy_scores_for_meta["strategy_source"].isin(ENSEMBLE_STRATEGY_SOURCES)
    & pd.to_datetime(base_strategy_scores_for_meta["date"], errors="coerce").ge(pd.Timestamp(META_OOS_START))
].copy()

meta_labeled_proba = meta_classifier.predict_proba_frame(meta_feature_panel_train, meta_feature_cols)
meta_strategy_scores_labeled = build_strategy_score_frame(
    source="stacked",
    family="meta",
    prediction_frame=meta_feature_panel_train,
    probability_frame=meta_labeled_proba,
    apply_ae_to_exits=False,
)
meta_strategy_scores_labeled["strategy_source"] = "stacked_meta"
meta_strategy_scores_labeled["source"] = "stacked"
meta_strategy_scores_labeled["family"] = "meta"
meta_strategy_scores_labeled_oos = meta_strategy_scores_labeled.loc[
    pd.to_datetime(meta_strategy_scores_labeled["date"], errors="coerce").ge(pd.Timestamp(META_OOS_START))
].copy()

comparison_labels_oos = oracle_label_rows.loc[pd.to_datetime(oracle_label_rows["date"], errors="coerce").ge(pd.Timestamp(META_OOS_START))].copy()
classification_comparison = pd.DataFrame(
    [
        _diagnose_score_frame_against_labels(
            base_ensemble_labeled_oos.loc[base_ensemble_labeled_oos["strategy_source"].eq(strategy_source)],
            comparison_labels_oos,
            name=strategy_source,
        )
        for strategy_source in ENSEMBLE_STRATEGY_SOURCES
    ]
    + [_diagnose_score_frame_against_labels(meta_strategy_scores_labeled_oos, comparison_labels_oos, name="stacked_meta")]
)

print("Secondary OOS label classification comparison")
display(classification_comparison)


## Primary Trading Performance Comparison

In [ ]:
from quant_orchestrator.research_tools.ml_trading_experiment import _load_price_frames
from quant_warehouse.warehouse.api import Warehouse


def _threshold_positions(score_wide: pd.DataFrame, *, threshold: float) -> pd.DataFrame:
    positions = pd.DataFrame(False, index=score_wide.index, columns=score_wide.columns)
    for symbol in score_wide.columns:
        active = False
        values = pd.to_numeric(score_wide[symbol], errors="coerce")
        out = []
        for value in values:
            if pd.notna(value):
                if float(value) > threshold:
                    active = True
                elif float(value) < threshold:
                    active = False
            out.append(active)
        positions[symbol] = out
    return positions


def run_simple_threshold_backtest(
    strategy_scores: pd.DataFrame,
    price_frames: dict[str, pd.DataFrame],
    *,
    threshold: float,
    start: str,
    cost_bps: float = 0.0,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if strategy_scores.empty or not price_frames:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    scores = strategy_scores.copy()
    scores["date"] = pd.to_datetime(scores["date"], errors="coerce").dt.normalize()
    scores["symbol"] = scores["symbol"].astype(str).str.upper()
    scores["long_score"] = pd.to_numeric(scores["long_score"], errors="coerce")
    wide_close = pd.DataFrame({symbol: frame["close"] for symbol, frame in price_frames.items()}).sort_index().ffill()
    next_returns = wide_close.pct_change().shift(-1)
    rows = []
    yearly_rows = []
    trade_rows = []
    for strategy_source, frame in scores.groupby("strategy_source", sort=True):
        score_wide = frame.pivot_table(index="date", columns="symbol", values="long_score", aggfunc="mean").sort_index()
        dates = pd.DatetimeIndex(sorted(set(score_wide.index).intersection(next_returns.index)))
        dates = dates[dates >= pd.Timestamp(start)]
        symbols = sorted(set(score_wide.columns).intersection(next_returns.columns))
        if len(dates) == 0 or not symbols:
            continue
        score_wide = score_wide.reindex(index=dates, columns=symbols)
        raw_positions = _threshold_positions(score_wide, threshold=threshold).astype("float64")
        active_counts = raw_positions.sum(axis=1).replace(0.0, np.nan)
        weights = raw_positions.div(active_counts, axis=0).fillna(0.0)
        aligned_returns = next_returns.reindex(index=dates, columns=symbols).fillna(0.0)
        gross_returns = (weights * aligned_returns).sum(axis=1)
        turnover = weights.diff().abs().sum(axis=1).fillna(weights.abs().sum(axis=1))
        net_returns = gross_returns - turnover * (float(cost_bps) / 10_000.0)
        equity_curve = (1.0 + net_returns).cumprod()
        total_return = float(equity_curve.iloc[-1] - 1.0) if len(equity_curve) else np.nan
        annualized_return = float(equity_curve.iloc[-1] ** (252.0 / max(1, len(net_returns))) - 1.0) if len(equity_curve) else np.nan
        annualized_vol = float(net_returns.std(ddof=0) * np.sqrt(252.0)) if len(net_returns) else np.nan
        sharpe = float(annualized_return / annualized_vol) if annualized_vol and np.isfinite(annualized_vol) and annualized_vol > 0 else np.nan
        drawdown = equity_curve / equity_curve.cummax() - 1.0 if len(equity_curve) else pd.Series(dtype="float64")
        rows.append(
            {
                "strategy_source": strategy_source,
                "rule": f"buy long_score > {threshold}; sell long_score < {threshold}",
                "threshold": threshold,
                "cost_bps": cost_bps,
                "start": str(pd.Timestamp(start).date()),
                "score_rows": len(frame),
                "score_symbols": len(symbols),
                "score_dates": len(dates),
                "active_days": int(weights.abs().sum(axis=1).gt(0).sum()),
                "avg_positions": float(raw_positions.sum(axis=1).mean()),
                "avg_gross_exposure": float(weights.abs().sum(axis=1).mean()),
                "avg_turnover": float(turnover.mean()),
                "total_return": total_return,
                "annualized_return": annualized_return,
                "annualized_vol": annualized_vol,
                "sharpe": sharpe,
                "max_drawdown": float(drawdown.min()) if len(drawdown) else np.nan,
                "win_rate": float(net_returns.gt(0).mean()) if len(net_returns) else np.nan,
            }
        )
        for year, returns in net_returns.groupby(net_returns.index.year):
            year_curve = (1.0 + returns).cumprod()
            yearly_rows.append(
                {
                    "strategy_source": strategy_source,
                    "year": int(year),
                    "days": int(len(returns)),
                    "total_return": float(year_curve.iloc[-1] - 1.0) if len(year_curve) else np.nan,
                    "annualized_vol": float(returns.std(ddof=0) * np.sqrt(252.0)) if len(returns) else np.nan,
                    "sharpe": float((returns.mean() * 252.0) / (returns.std(ddof=0) * np.sqrt(252.0))) if returns.std(ddof=0) > 0 else np.nan,
                    "active_days": int(weights.loc[returns.index].abs().sum(axis=1).gt(0).sum()),
                    "avg_positions": float(raw_positions.loc[returns.index].sum(axis=1).mean()),
                }
            )
        previous = pd.DataFrame(0.0, index=weights.index, columns=weights.columns).shift(1).fillna(0.0)
        delta = weights - previous
        for date, changes in delta.iterrows():
            changed = changes.loc[changes.ne(0)]
            for symbol, weight_delta in changed.items():
                trade_rows.append(
                    {
                        "date": date,
                        "strategy_source": strategy_source,
                        "symbol": symbol,
                        "action": "buy" if weight_delta > 0 else "sell",
                        "target_weight": float(weights.at[date, symbol]),
                        "weight_delta": float(weight_delta),
                        "long_score": float(score_wide.at[date, symbol]) if pd.notna(score_wide.at[date, symbol]) else np.nan,
                    }
                )
    return pd.DataFrame(rows), pd.DataFrame(trade_rows), pd.DataFrame(yearly_rows)


def _compact_score_columns(frame: pd.DataFrame) -> pd.DataFrame:
    keep = [
        "strategy_source",
        "source",
        "family",
        "symbol",
        "date",
        "long_score",
        "short_score",
        "long_exit_score",
        "short_exit_score",
        "net_score",
        "model_count",
        "long_score_rank",
        "short_score_rank",
        "net_score_rank",
    ]
    out = frame[[col for col in keep if col in frame.columns]].copy()
    out["symbol"] = out["symbol"].astype(str).str.upper()
    out["date"] = pd.to_datetime(out["date"], errors="coerce").dt.normalize()
    return out


def score_daily_chunks_for_backtest() -> pd.DataFrame:
    date_bounds = []
    for row in feature_family_index.to_dict("records"):
        path = Path(row["panel_path"])
        if not path.exists() or int(row.get("rows") or 0) <= 0:
            continue
        dates = pd.read_parquet(path, columns=["date"])
        dates["date"] = pd.to_datetime(dates["date"], errors="coerce").dt.normalize()
        dates = dates.loc[dates["date"].ge(pd.Timestamp(META_OOS_START)), "date"].dropna()
        if not dates.empty:
            date_bounds.extend(dates.dt.year.astype(int).unique().tolist())
    years = sorted(set(date_bounds))
    chunks = []
    for year in years:
        family_cache_name = f"daily_family_scores_{year}"
        family_manifest = _family_score_cache_manifest(
            cache_name=family_cache_name,
            score_scope="full_daily_universe",
            train_end=BASE_TRAIN_END,
            score_start=f"{year}-01-01",
            score_end=str(min(pd.Timestamp(f"{year}-12-31"), pd.Timestamp.today().normalize()).date()),
            score_year=int(year),
        )
        base_single_scores_chunk = _load_family_score_cache(family_cache_name, family_manifest)
        if base_single_scores_chunk is None:
            family_score_frames = []
            print(f"[daily-score] scoring {year} family-by-family", flush=True)
            for (source, family), payload in trained_models.items():
                panel = load_feature_family_panel(source, family)
                if panel.empty:
                    continue
                panel = panel.loc[pd.to_datetime(panel["date"], errors="coerce").dt.year.eq(year)].copy()
                if panel.empty:
                    continue
                scores = score_trained_family_panel(
                    panel,
                    payload,
                    source=str(source),
                    family=str(family),
                    score_start=pd.Timestamp(f"{year}-01-01"),
                    score_keys=None,
                )
                if not scores.empty:
                    family_score_frames.append(scores)
                del panel, scores
                gc.collect()
            base_single_scores_chunk = pd.concat(family_score_frames, ignore_index=True, sort=False) if family_score_frames else pd.DataFrame()
            if base_single_scores_chunk.empty:
                continue
            _write_family_score_cache(family_cache_name, base_single_scores_chunk, family_manifest)
            del family_score_frames
        else:
            print(f"[daily-score] using cached raw family scores for {year}; recomputing ensemble/meta views")
        ensemble_chunk = build_ensemble_mean_scores(base_single_scores_chunk)
        base_scores_chunk = pd.concat([ensemble_chunk, base_single_scores_chunk], ignore_index=True, sort=False)
        meta_feature_panel_chunk, _meta_chunk_cols = build_meta_feature_panel(base_scores_chunk)
        for col in meta_feature_cols:
            if col not in meta_feature_panel_chunk.columns:
                meta_feature_panel_chunk[col] = 0.0
        meta_feature_panel_chunk = meta_feature_panel_chunk[["symbol", "date", *meta_feature_cols]].copy()
        meta_daily_proba = meta_classifier.predict_proba_frame(meta_feature_panel_chunk, meta_feature_cols)
        meta_scores_chunk = build_strategy_score_frame(
            source="stacked",
            family="meta",
            prediction_frame=meta_feature_panel_chunk,
            probability_frame=meta_daily_proba,
            apply_ae_to_exits=False,
        )
        meta_scores_chunk["strategy_source"] = "stacked_meta"
        meta_scores_chunk["source"] = "stacked"
        meta_scores_chunk["family"] = "meta"
        chunk_scores = pd.concat(
            [_compact_score_columns(ensemble_chunk), _compact_score_columns(meta_scores_chunk)],
            ignore_index=True,
            sort=False,
        )
        print(f"[daily-score] prepared {year}: rows={len(chunk_scores):,} from family_rows={len(base_single_scores_chunk):,}")
        chunks.append(chunk_scores)
        del base_single_scores_chunk, base_scores_chunk, meta_feature_panel_chunk, meta_scores_chunk, ensemble_chunk, chunk_scores
        gc.collect()
    return pd.concat(chunks, ignore_index=True, sort=False) if chunks else pd.DataFrame()

price_warehouse = Warehouse()
price_frames = _load_price_frames(
    price_warehouse,
    screened_equity_symbols,
    provider="fmp",
    start=DATA_START,
    end=optional_date(DATA_END),
)

backtest_score_started = perf_counter()
comparison_strategy_scores = score_daily_chunks_for_backtest()
print(
    {
        "daily_scoring_seconds": round(perf_counter() - backtest_score_started, 2),
        "comparison_score_rows": len(comparison_strategy_scores),
        "comparison_score_symbols": comparison_strategy_scores["symbol"].nunique() if not comparison_strategy_scores.empty else 0,
    }
)

backtest_started = perf_counter()
backtest_summary, trade_log, yearly_backtest_summary = run_simple_threshold_backtest(
    comparison_strategy_scores,
    price_frames,
    threshold=SCORE_THRESHOLD,
    start=META_OOS_START,
    cost_bps=SIMPLE_RULE_COST_BPS,
)
print({"backtest_seconds": round(perf_counter() - backtest_started, 2)})
if backtest_summary.empty:
    print("Simple threshold backtest summary is empty. No aligned prices/scores were available.")
    trading_performance_leaderboard = pd.DataFrame()
else:
    trading_performance_leaderboard = backtest_summary.sort_values(["sharpe", "total_return"], ascending=[False, False]).reset_index(drop=True)
    print("Primary result: post-2021 simple threshold trading performance")
    display(trading_performance_leaderboard)
print("Yearly simple threshold trading summary")
display(yearly_backtest_summary.sort_values(["strategy_source", "year"]) if not yearly_backtest_summary.empty else yearly_backtest_summary)


## Anchored Walk-Forward Portfolio Test

Each forward year retrains the feature-family classifiers and stacked meta classifier on all oracle-label data available before that year. The forward test then evaluates the same scores with `top_k=None, 5, 10, 20, 40`.


In [ ]:
def run_top_k_entry_gate_backtest(
    strategy_scores: pd.DataFrame,
    price_frames: dict[str, pd.DataFrame],
    *,
    threshold: float,
    start: str | pd.Timestamp,
    end: str | pd.Timestamp | None = None,
    top_k_values: tuple[object, ...] = (None, 5, 10, 20, 40),
    cost_bps: float = 0.0,
    forward_year: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if strategy_scores.empty or not price_frames:
        return pd.DataFrame(), pd.DataFrame()
    scores = strategy_scores.copy()
    scores["date"] = pd.to_datetime(scores["date"], errors="coerce").dt.normalize()
    scores["symbol"] = scores["symbol"].astype(str).str.upper()
    scores["long_score"] = pd.to_numeric(scores["long_score"], errors="coerce")
    start_ts = pd.Timestamp(start).normalize()
    end_ts = pd.Timestamp(end).normalize() if end is not None else None

    wide_close = pd.DataFrame({symbol: frame["close"] for symbol, frame in price_frames.items()}).sort_index().ffill()
    wide_close.index = pd.to_datetime(wide_close.index, errors="coerce").normalize()
    next_returns = wide_close.pct_change().shift(-1)
    summary_rows = []
    daily_rows = []

    for strategy_source, frame in scores.groupby("strategy_source", sort=True):
        score_wide = frame.pivot_table(index="date", columns="symbol", values="long_score", aggfunc="mean").sort_index()
        dates = pd.DatetimeIndex(sorted(set(score_wide.index).intersection(next_returns.index)))
        dates = dates[dates >= start_ts]
        if end_ts is not None:
            dates = dates[dates <= end_ts]
        symbols = sorted(set(score_wide.columns).intersection(next_returns.columns))
        if len(dates) == 0 or not symbols:
            continue
        score_wide = score_wide.reindex(index=dates, columns=symbols)
        aligned_returns = next_returns.reindex(index=dates, columns=symbols).fillna(0.0)

        for top_k in top_k_values:
            raw_positions = pd.DataFrame(0.0, index=dates, columns=symbols)
            active: set[str] = set()
            for date in dates:
                day_scores = pd.to_numeric(score_wide.loc[date], errors="coerce")
                for symbol in list(active):
                    value = day_scores.get(symbol, np.nan)
                    if pd.notna(value) and float(value) < threshold:
                        active.remove(symbol)
                if top_k is None:
                    candidates = day_scores.loc[day_scores.gt(threshold)].sort_values(ascending=False)
                else:
                    candidates = day_scores.sort_values(ascending=False).head(int(top_k))
                    candidates = candidates.loc[candidates.gt(threshold)]
                for symbol in candidates.index.astype(str):
                    if top_k is not None and len(active) >= int(top_k):
                        break
                    active.add(symbol)
                if active:
                    raw_positions.loc[date, sorted(active)] = 1.0
            active_counts = raw_positions.sum(axis=1).replace(0.0, np.nan)
            weights = raw_positions.div(active_counts, axis=0).fillna(0.0)
            gross_returns = (weights * aligned_returns).sum(axis=1)
            turnover = weights.diff().abs().sum(axis=1).fillna(weights.abs().sum(axis=1))
            net_returns = gross_returns - turnover * (float(cost_bps) / 10_000.0)
            equity_curve = (1.0 + net_returns).cumprod()
            drawdown = equity_curve / equity_curve.cummax() - 1.0 if len(equity_curve) else pd.Series(dtype="float64")
            annualized_return = float(equity_curve.iloc[-1] ** (252.0 / max(1, len(net_returns))) - 1.0) if len(equity_curve) else np.nan
            annualized_vol = float(net_returns.std(ddof=0) * np.sqrt(252.0)) if len(net_returns) else np.nan
            summary_rows.append(
                {
                    "forward_year": forward_year,
                    "strategy_source": strategy_source,
                    "top_k": top_k,
                    "days": int(len(net_returns)),
                    "active_days": int(weights.abs().sum(axis=1).gt(0).sum()),
                    "avg_positions": float(raw_positions.sum(axis=1).mean()),
                    "median_positions": float(raw_positions.sum(axis=1).median()),
                    "avg_gross_exposure": float(weights.abs().sum(axis=1).mean()),
                    "avg_turnover": float(turnover.mean()),
                    "total_return": float(equity_curve.iloc[-1] - 1.0) if len(equity_curve) else np.nan,
                    "annualized_return": annualized_return,
                    "annualized_vol": annualized_vol,
                    "sharpe": float(annualized_return / annualized_vol) if annualized_vol and np.isfinite(annualized_vol) and annualized_vol > 0 else np.nan,
                    "max_drawdown": float(drawdown.min()) if len(drawdown) else np.nan,
                    "win_rate": float(net_returns.gt(0).mean()) if len(net_returns) else np.nan,
                }
            )
            daily_rows.extend(
                {
                    "date": date,
                    "forward_year": forward_year,
                    "strategy_source": strategy_source,
                    "top_k": top_k,
                    "net_return": float(value),
                    "positions": float(raw_positions.loc[date].sum()),
                    "turnover": float(turnover.loc[date]),
                }
                for date, value in net_returns.items()
            )
    return pd.DataFrame(summary_rows), pd.DataFrame(daily_rows)


def _train_wfo_family_models(train_cutoff: pd.Timestamp) -> tuple[dict[tuple[str, str], dict], pd.DataFrame, pd.DataFrame]:
    model_rows = []
    trained_fold_models: dict[tuple[str, str], dict] = {}
    meta_score_frames = []
    oracle_train_keys = oracle_label_rows.loc[
        pd.to_datetime(oracle_label_rows["date"], errors="coerce").le(train_cutoff),
        ["symbol", "date"],
    ].drop_duplicates().copy()
    oracle_train_keys["symbol"] = oracle_train_keys["symbol"].astype(str).str.upper()
    oracle_train_keys["date"] = pd.to_datetime(oracle_train_keys["date"], errors="coerce").dt.normalize()

    training_plan = feature_family_index.loc[feature_family_index["features"].gt(0)].sort_values(["source", "family"])
    for job_index, row in enumerate(training_plan.to_dict("records"), start=1):
        source = str(row["source"])
        family = str(row["family"])
        strategy_source = f"{source}.{family}"
        panel = load_feature_family_panel(source, family)
        metadata = selected_feature_metadata.loc[
            selected_feature_metadata["source"].astype(str).eq(source)
            & selected_feature_metadata["family"].astype(str).eq(family)
        ].copy()
        family_frame, features = prepare_family_dataset(
            panel,
            metadata,
            oracle_label_rows,
            source=source,
            family=family,
            min_feature_coverage=base_config.min_feature_coverage,
        )
        family_frame["date"] = pd.to_datetime(family_frame["date"], errors="coerce").dt.normalize()
        train = family_frame.loc[family_frame["date"].le(train_cutoff)].copy()
        print(
            f"[anchored-wfo] train <= {train_cutoff.date()} {job_index}/{len(training_plan)} "
            f"{strategy_source} rows={len(train):,} features={len(features):,}",
            flush=True,
        )
        if len(train) < base_config.min_train_rows_per_family or train["collapsed_label"].nunique() < base_config.min_classes_per_family:
            model_rows.append(
                {
                    "strategy_source": strategy_source,
                    "status": "skipped_sparse_train",
                    "features": len(features),
                    "train_rows": len(train),
                    "regime_year_feature_included": bool(RUN_REGIME_YEAR_FEATURE and REGIME_YEAR_FEATURE in features),
                    "regime_year_feature_importance": np.nan,
                }
            )
            del panel, metadata, family_frame, train
            gc.collect()
            continue
        started = perf_counter()
        classifier = RapidsRandomForestClassifier.fit(
            train,
            features=features,
            target_col="collapsed_label",
            random_state=RANDOM_SEED,
            params=BASE_RF_PARAMS,
        )
        payload = {
            "classifier": classifier,
            "autoencoder": None,
            "feature_autoencoder": None,
            "features": features,
            "raw_features": features,
            "base_family": family,
            "representation": "raw",
        }
        trained_fold_models[(source, family)] = payload
        model_rows.append(
            {
                "strategy_source": strategy_source,
                "status": "ok",
                "features": len(features),
                "train_rows": len(train),
                "regime_year_feature_included": bool(RUN_REGIME_YEAR_FEATURE and REGIME_YEAR_FEATURE in features),
                "regime_year_feature_importance": regime_year_feature_importance(classifier, features),
                "fit_seconds": perf_counter() - started,
            }
        )
        scores = score_trained_family_panel(
            panel,
            payload,
            source=source,
            family=family,
            score_start=pd.Timestamp(DATA_START),
            score_keys=oracle_train_keys,
        )
        if not scores.empty:
            meta_score_frames.append(scores)
        del panel, metadata, family_frame, train, scores
        gc.collect()
    base_single_scores = pd.concat(meta_score_frames, ignore_index=True, sort=False) if meta_score_frames else pd.DataFrame()
    return trained_fold_models, base_single_scores, pd.DataFrame(model_rows)


def _score_wfo_forward_year(
    trained_fold_models: dict[tuple[str, str], dict],
    meta_classifier_fold,
    meta_feature_cols_fold: list[str],
    forward_year: int,
) -> pd.DataFrame:
    family_score_frames = []
    for (source, family), payload in trained_fold_models.items():
        panel = load_feature_family_panel(source, family)
        if panel.empty:
            continue
        panel = panel.loc[pd.to_datetime(panel["date"], errors="coerce").dt.year.eq(forward_year)].copy()
        if panel.empty:
            continue
        scores = score_trained_family_panel(
            panel,
            payload,
            source=source,
            family=family,
            score_start=pd.Timestamp(f"{forward_year}-01-01"),
            score_keys=None,
        )
        if not scores.empty:
            family_score_frames.append(scores)
        del panel, scores
        gc.collect()
    base_single_scores = pd.concat(family_score_frames, ignore_index=True, sort=False) if family_score_frames else pd.DataFrame()
    if base_single_scores.empty:
        return pd.DataFrame()
    ensemble_scores = build_ensemble_mean_scores(base_single_scores)
    base_scores = pd.concat([ensemble_scores, base_single_scores], ignore_index=True, sort=False)
    meta_feature_panel_forward, _ = build_meta_feature_panel(base_scores)
    for col in meta_feature_cols_fold:
        if col not in meta_feature_panel_forward.columns:
            meta_feature_panel_forward[col] = 0.0
    meta_feature_panel_forward = meta_feature_panel_forward[["symbol", "date", *meta_feature_cols_fold]].copy()
    meta_proba = meta_classifier_fold.predict_proba_frame(meta_feature_panel_forward, meta_feature_cols_fold)
    meta_scores = build_strategy_score_frame(
        source="stacked",
        family="meta",
        prediction_frame=meta_feature_panel_forward,
        probability_frame=meta_proba,
        apply_ae_to_exits=False,
    )
    meta_scores["strategy_source"] = "stacked_meta"
    meta_scores["source"] = "stacked"
    meta_scores["family"] = "meta"
    score_parts = [_compact_score_columns(ensemble_scores), _compact_score_columns(meta_scores)]
    if INCLUDE_FAMILY_STRATEGY_WFO:
        score_parts.append(_compact_score_columns(base_single_scores))
    return pd.concat(score_parts, ignore_index=True, sort=False)


def _summarize_wfo_outputs(fold_summary: pd.DataFrame, daily_returns: pd.DataFrame) -> pd.DataFrame:
    if fold_summary.empty:
        return pd.DataFrame()
    rows = []
    for (strategy_source, top_k), fold_group in fold_summary.groupby(["strategy_source", "top_k"], dropna=False):
        daily_group = daily_returns.loc[
            daily_returns["strategy_source"].eq(strategy_source)
            & (daily_returns["top_k"].isna() if pd.isna(top_k) else daily_returns["top_k"].eq(top_k))
        ].sort_values("date")
        if daily_group.empty:
            stitched_return = float(np.prod(1.0 + fold_group["total_return"].astype(float)) - 1.0)
            stitched_sharpe = np.nan
            stitched_max_drawdown = np.nan
        else:
            net_returns = pd.Series(daily_group["net_return"].astype(float).to_numpy(), index=pd.to_datetime(daily_group["date"]))
            equity_curve = (1.0 + net_returns).cumprod()
            stitched_return = float(equity_curve.iloc[-1] - 1.0)
            ann_return = float(equity_curve.iloc[-1] ** (252.0 / max(1, len(net_returns))) - 1.0)
            ann_vol = float(net_returns.std(ddof=0) * np.sqrt(252.0))
            stitched_sharpe = float(ann_return / ann_vol) if ann_vol > 0 else np.nan
            stitched_max_drawdown = float((equity_curve / equity_curve.cummax() - 1.0).min())
        rows.append(
            {
                "strategy_source": strategy_source,
                "top_k": top_k,
                "folds": int(fold_group["forward_year"].nunique()),
                "stitched_total_return": stitched_return,
                "stitched_sharpe": stitched_sharpe,
                "stitched_max_drawdown": stitched_max_drawdown,
                "mean_annual_return": float(fold_group["total_return"].mean()),
                "median_annual_return": float(fold_group["total_return"].median()),
                "mean_fold_sharpe": float(fold_group["sharpe"].mean()),
                "worst_year_return": float(fold_group["total_return"].min()),
                "mean_max_drawdown": float(fold_group["max_drawdown"].mean()),
                "mean_avg_positions": float(fold_group["avg_positions"].mean()),
                "mean_turnover": float(fold_group["avg_turnover"].mean()),
            }
        )
    return pd.DataFrame(rows).sort_values(["stitched_sharpe", "stitched_total_return"], ascending=[False, False]).reset_index(drop=True)


anchored_wfo_dir = RUN_ARTIFACT_DIR / ANCHORED_WFO_OUTPUT_NAME
anchored_wfo_dir.mkdir(parents=True, exist_ok=True)
anchored_wfo_fold_frames = []
anchored_wfo_daily_frames = []
anchored_wfo_fit_frames = []

if not RUN_ANCHORED_WFO:
    print("Skipping anchored WFO because RUN_ANCHORED_WFO=False")
    anchored_wfo_fold_summary = pd.DataFrame()
    anchored_wfo_daily_returns = pd.DataFrame()
    anchored_wfo_stitched_summary = pd.DataFrame()
else:
    wfo_started = perf_counter()
    for forward_year in range(pd.Timestamp(META_OOS_START).year, pd.Timestamp.today().year + 1):
        train_cutoff = pd.Timestamp(f"{forward_year - 1}-12-31")
        forward_start = pd.Timestamp(f"{forward_year}-01-01")
        forward_end = min(pd.Timestamp(f"{forward_year}-12-31"), pd.Timestamp.today().normalize())
        fold_scores_path = anchored_wfo_dir / f"fold_{forward_year}_scores.pkl"
        fold_summary_path = anchored_wfo_dir / f"fold_{forward_year}_summary.csv"
        fold_daily_path = anchored_wfo_dir / f"fold_{forward_year}_daily_returns.csv"
        if REUSE_ANCHORED_WFO_FOLDS and fold_scores_path.exists() and fold_summary_path.exists() and fold_daily_path.exists():
            fold_scores = pd.read_pickle(fold_scores_path)
            fold_summary = pd.read_csv(fold_summary_path)
            fold_daily = pd.read_csv(fold_daily_path)
            print(f"[anchored-wfo] loaded cached fold {forward_year}: rows={len(fold_scores):,}")
        else:
            print(f"[anchored-wfo] fold {forward_year}: train <= {train_cutoff.date()}, forward {forward_start.date()}..{forward_end.date()}")
            trained_fold_models, base_single_scores_train, fit_rows = _train_wfo_family_models(train_cutoff)
            base_scores_train = pd.concat(
                [build_ensemble_mean_scores(base_single_scores_train), base_single_scores_train],
                ignore_index=True,
                sort=False,
            ) if not base_single_scores_train.empty else pd.DataFrame()
            meta_feature_panel_fold, meta_feature_cols_fold = build_meta_feature_panel(base_scores_train)
            meta_dataset_fold = oracle_label_rows.merge(meta_feature_panel_fold, on=["symbol", "date"], how="inner")
            meta_dataset_fold["date"] = pd.to_datetime(meta_dataset_fold["date"], errors="coerce").dt.normalize()
            meta_train_fold = meta_dataset_fold.loc[meta_dataset_fold["date"].le(train_cutoff)].copy()
            if len(meta_train_fold) < 250 or meta_train_fold["collapsed_label"].nunique() < 2:
                raise RuntimeError(
                    f"Not enough meta training rows/classes for WFO {forward_year}: "
                    f"rows={len(meta_train_fold)}, classes={meta_train_fold['collapsed_label'].nunique()}"
                )
            print(f"[anchored-wfo] train meta {forward_year}: rows={len(meta_train_fold):,} features={len(meta_feature_cols_fold):,}")
            meta_classifier_fold = RapidsRandomForestClassifier.fit(
                meta_train_fold,
                features=meta_feature_cols_fold,
                target_col="collapsed_label",
                random_state=RANDOM_SEED,
                params=META_RF_PARAMS,
            )
            fold_scores = _score_wfo_forward_year(trained_fold_models, meta_classifier_fold, meta_feature_cols_fold, forward_year)
            fold_summary, fold_daily = run_top_k_entry_gate_backtest(
                fold_scores,
                price_frames,
                threshold=SCORE_THRESHOLD,
                start=forward_start,
                end=forward_end,
                top_k_values=ANCHORED_WFO_TOP_K_VALUES,
                cost_bps=SIMPLE_RULE_COST_BPS,
                forward_year=forward_year,
            )
            fit_rows.insert(0, "forward_year", forward_year)
            fold_scores.to_pickle(fold_scores_path)
            fold_summary.to_csv(fold_summary_path, index=False)
            fold_daily.to_csv(fold_daily_path, index=False)
            anchored_wfo_fit_frames.append(fit_rows)
            del trained_fold_models, base_single_scores_train, base_scores_train, meta_feature_panel_fold, meta_dataset_fold, meta_train_fold, meta_classifier_fold
            gc.collect()
        anchored_wfo_fold_frames.append(fold_summary)
        anchored_wfo_daily_frames.append(fold_daily)

    anchored_wfo_fold_summary = pd.concat(anchored_wfo_fold_frames, ignore_index=True, sort=False) if anchored_wfo_fold_frames else pd.DataFrame()
    anchored_wfo_daily_returns = pd.concat(anchored_wfo_daily_frames, ignore_index=True, sort=False) if anchored_wfo_daily_frames else pd.DataFrame()
    anchored_wfo_stitched_summary = _summarize_wfo_outputs(anchored_wfo_fold_summary, anchored_wfo_daily_returns)
    anchored_wfo_fold_summary.to_csv(anchored_wfo_dir / "anchored_wfo_fold_summary.csv", index=False)
    anchored_wfo_daily_returns.to_csv(anchored_wfo_dir / "anchored_wfo_daily_returns.csv", index=False)
    anchored_wfo_stitched_summary.to_csv(anchored_wfo_dir / "anchored_wfo_stitched_summary.csv", index=False)
    if anchored_wfo_fit_frames:
        pd.concat(anchored_wfo_fit_frames, ignore_index=True, sort=False).to_csv(anchored_wfo_dir / "model_fit_rows.csv", index=False)
    print({"anchored_wfo_seconds": round(perf_counter() - wfo_started, 2), "anchored_wfo_dir": str(anchored_wfo_dir)})
    family_strategy_wfo_leaderboard = anchored_wfo_stitched_summary.loc[
        ~anchored_wfo_stitched_summary["strategy_source"].isin([*ENSEMBLE_STRATEGY_SOURCES, "stacked_meta"])
    ].copy()
    family_strategy_wfo_leaderboard.to_csv(anchored_wfo_dir / "family_strategy_wfo_leaderboard.csv", index=False)
    print("Anchored WFO stitched summary")
    display(anchored_wfo_stitched_summary)
    print("Family strategy WFO leaderboard")
    display(family_strategy_wfo_leaderboard)
    print("Anchored WFO fold summary")
    display(anchored_wfo_fold_summary.sort_values(["forward_year", "strategy_source", "top_k"], na_position="first"))


## Symbol-Level Backtesting.py Robustness

Run each strategy independently on each symbol with the same simple rule: buy when `long_score > 0.5`, sell when `long_score < 0.5`. The key metric is the percentage of symbols where the strategy beats buy-and-hold for that same symbol over the post-2021 window.


In [ ]:
from backtesting import Backtest
from backtesting.lib import SignalStrategy

MAG7_SYMBOLS = {"AAPL", "AMZN", "GOOG", "GOOGL", "META", "MSFT", "NVDA", "TSLA"}


def _single_symbol_threshold_signal_frame(
    prices: pd.DataFrame,
    scores: pd.DataFrame,
    *,
    threshold: float,
) -> pd.DataFrame:
    frame = prices.rename(columns=str.lower).copy()
    required = ["open", "high", "low", "close", "volume"]
    missing = set(required).difference(frame.columns)
    if missing:
        raise KeyError(f"prices missing required columns: {sorted(missing)}")
    frame = frame[required].apply(pd.to_numeric, errors="coerce")
    frame.index = pd.DatetimeIndex(pd.to_datetime(frame.index, errors="coerce")).normalize()
    frame = frame.dropna(subset=["open", "high", "low", "close"]).sort_index()

    score_frame = scores.copy()
    score_frame["date"] = pd.to_datetime(score_frame["date"], errors="coerce").dt.normalize()
    score_frame["long_score"] = pd.to_numeric(score_frame["long_score"], errors="coerce")
    score_series = (
        score_frame.dropna(subset=["date"])
        .drop_duplicates("date")
        .set_index("date")
        .sort_index()["long_score"]
        .reindex(frame.index)
    )

    active = False
    positions = []
    for value in score_series:
        if pd.notna(value):
            if float(value) > threshold:
                active = True
            elif float(value) < threshold:
                active = False
        positions.append(1.0 if active else 0.0)
    position = pd.Series(positions, index=frame.index, dtype="float64")
    previous = position.shift(1).fillna(0.0)
    entry_size = pd.Series(0.0, index=frame.index)
    exit_portion = pd.Series(0.0, index=frame.index)
    entry_size.loc[(previous.eq(0.0)) & position.gt(0.0)] = 0.999
    exit_portion.loc[(previous.gt(0.0)) & position.eq(0.0)] = 1.0

    out = frame.rename(columns={"open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"})
    out["signal_entry_size"] = entry_size.shift(1).fillna(0.0)
    out["signal_exit_portion"] = exit_portion.shift(1).fillna(0.0)
    out["long_score"] = score_series
    return out.dropna(subset=["Open", "High", "Low", "Close"])


def run_backtesting_py_symbol_threshold(
    prices: pd.DataFrame,
    scores: pd.DataFrame,
    *,
    symbol: str,
    strategy_source: str,
    threshold: float,
    cash: float = 100_000.0,
    commission_bps: float = 0.5,
    spread_bps: float = 5.0,
) -> dict[str, object]:
    data = _single_symbol_threshold_signal_frame(prices, scores, threshold=threshold)
    if len(data) < 3:
        return {
            "framework": "backtesting.py",
            "strategy_source": strategy_source,
            "symbol": symbol,
            "status": "skipped_short_history",
            "days": len(data),
        }

    class ThresholdSignalStrategy(SignalStrategy):
        def init(self):
            super().init()
            self.set_signal(self.data.signal_entry_size, self.data.signal_exit_portion, plot=False)

    stats = Backtest(
        data,
        ThresholdSignalStrategy,
        cash=float(cash),
        commission=float(commission_bps) / 10_000.0,
        spread=float(spread_bps) / 10_000.0,
        trade_on_close=False,
        exclusive_orders=True,
        finalize_trades=True,
    ).run()
    strategy_return = float(stats.get("Return [%]", np.nan)) / 100.0
    buy_hold_return = float(data["Close"].iloc[-1] / data["Close"].iloc[0] - 1.0) if len(data) > 1 else np.nan
    return {
        "framework": "backtesting.py",
        "strategy_source": strategy_source,
        "symbol": str(symbol).upper(),
        "is_mag7": str(symbol).upper() in MAG7_SYMBOLS,
        "status": "ok",
        "rule": f"buy long_score > {threshold}; sell long_score < {threshold}",
        "days": int(len(data)),
        "trades": int(stats.get("# Trades", 0)),
        "strategy_total_return": strategy_return,
        "buy_hold_total_return": buy_hold_return,
        "excess_total_return": strategy_return - buy_hold_return,
        "beats_buy_hold": bool(strategy_return > buy_hold_return) if np.isfinite(strategy_return) and np.isfinite(buy_hold_return) else False,
        "strategy_sharpe": float(stats.get("Sharpe Ratio", np.nan)),
        "strategy_max_drawdown": float(stats.get("Max. Drawdown [%]", np.nan)) / 100.0,
        "strategy_win_rate": float(stats.get("Win Rate [%]", np.nan)) / 100.0,
    }


symbol_results_path = RUN_ARTIFACT_DIR / "symbol_level_backtesting_py.csv"
if not RUN_SYMBOL_LEVEL_BACKTESTING_PY:
    print("Skipping symbol-level backtesting.py because RUN_SYMBOL_LEVEL_BACKTESTING_PY=False")
    symbol_level_backtesting_py = pd.DataFrame()
    symbol_beat_summary = pd.DataFrame()
    symbol_beat_by_group = pd.DataFrame()
else:
    existing_symbol_results = (
        pd.read_csv(symbol_results_path)
        if REUSE_SYMBOL_LEVEL_BACKTESTS and symbol_results_path.exists()
        else pd.DataFrame()
    )
    symbol_rows = existing_symbol_results.to_dict("records") if not existing_symbol_results.empty else []
    completed_symbol_keys = set()
    if not existing_symbol_results.empty and {"strategy_source", "symbol"}.issubset(existing_symbol_results.columns):
        completed_symbol_keys = set(
            zip(
                existing_symbol_results["strategy_source"].astype(str),
                existing_symbol_results["symbol"].astype(str).str.upper(),
            )
        )
        print(f"Loaded {len(existing_symbol_results):,} existing symbol-level backtesting.py rows from {symbol_results_path}")

    pending_counter = 0
    for strategy_source, strategy_frame in comparison_strategy_scores.groupby("strategy_source", sort=True):
        for symbol, symbol_scores in strategy_frame.groupby("symbol", sort=True):
            symbol = str(symbol).upper()
            if (str(strategy_source), symbol) in completed_symbol_keys:
                continue
            prices = price_frames.get(symbol)
            if prices is None or prices.empty:
                continue
            price_index = pd.DatetimeIndex(pd.to_datetime(prices.index, errors="coerce"))
            price_window = prices.loc[price_index >= pd.Timestamp(META_OOS_START)].copy()
            if len(price_window) < 3:
                continue
            try:
                symbol_rows.append(
                    run_backtesting_py_symbol_threshold(
                        price_window,
                        symbol_scores,
                        symbol=symbol,
                        strategy_source=strategy_source,
                        threshold=SCORE_THRESHOLD,
                        cash=100_000.0,
                        commission_bps=0.5,
                        spread_bps=SIMPLE_RULE_COST_BPS - 0.5,
                    )
                )
            except Exception as exc:
                symbol_rows.append(
                    {
                        "framework": "backtesting.py",
                        "strategy_source": strategy_source,
                        "symbol": symbol,
                        "status": "error",
                        "error": repr(exc),
                    }
                )
            pending_counter += 1
            if pending_counter % int(SYMBOL_BACKTEST_CHECKPOINT_EVERY) == 0:
                pd.DataFrame(symbol_rows).to_csv(symbol_results_path, index=False)
                print(f"[symbol-backtesting.py] checkpoint rows={len(symbol_rows):,} path={symbol_results_path}")

    symbol_level_backtesting_py = pd.DataFrame(symbol_rows)
    if not symbol_level_backtesting_py.empty:
        symbol_level_backtesting_py.to_csv(symbol_results_path, index=False)

if symbol_level_backtesting_py.empty or "status" not in symbol_level_backtesting_py.columns:
    symbol_level_ok = pd.DataFrame()
else:
    symbol_level_ok = symbol_level_backtesting_py.loc[symbol_level_backtesting_py["status"].eq("ok")].copy()

if symbol_level_ok.empty:
    symbol_beat_summary = pd.DataFrame()
    symbol_beat_by_group = pd.DataFrame()
else:
    symbol_beat_summary = (
        symbol_level_ok.groupby("strategy_source", as_index=False)
        .agg(
            symbols_tested=("symbol", "nunique"),
            beat_count=("beats_buy_hold", "sum"),
            beat_rate=("beats_buy_hold", "mean"),
            mean_excess_return=("excess_total_return", "mean"),
            median_excess_return=("excess_total_return", "median"),
            mean_strategy_return=("strategy_total_return", "mean"),
            mean_buy_hold_return=("buy_hold_total_return", "mean"),
            median_trades=("trades", "median"),
        )
        .sort_values(["beat_rate", "median_excess_return"], ascending=[False, False])
        .reset_index(drop=True)
    )
    symbol_beat_by_group = (
        symbol_level_ok.assign(group=np.where(symbol_level_ok["is_mag7"], "mag7", "non_mag7"))
        .groupby(["strategy_source", "group"], as_index=False)
        .agg(
            symbols_tested=("symbol", "nunique"),
            beat_count=("beats_buy_hold", "sum"),
            beat_rate=("beats_buy_hold", "mean"),
            median_excess_return=("excess_total_return", "median"),
            mean_excess_return=("excess_total_return", "mean"),
        )
        .sort_values(["group", "beat_rate"], ascending=[True, False])
        .reset_index(drop=True)
    )

print("Percent of symbols where each strategy beats buy-and-hold")
display(symbol_beat_summary)
print("Beat rate split by Mag7 vs non-Mag7")
display(symbol_beat_by_group)
print("Best and worst symbol-level excess returns")
display(
    pd.concat(
        [
            symbol_level_ok.sort_values("excess_total_return", ascending=False).head(20),
            symbol_level_ok.sort_values("excess_total_return", ascending=True).head(20),
        ],
        ignore_index=True,
    ) if not symbol_level_ok.empty else symbol_level_ok
)


## Save Experiment Artifacts


In [ ]:
RUN_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_COMPARISON_DIR = META_ARTIFACT_DIR / "experiment_comparison"
EXPERIMENT_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_RESULT_ID = (
    f"{RUN_ARTIFACT_KEY}__events_{'-'.join(EVENT_LABEL_FAMILIES if INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS else ('oracle_only',))}"
    f"__rank_{int(INCLUDE_FAMILY_SCORE_RANK_FEATURES)}"
    f"__tech_{int(INCLUDE_TECHNICAL_FEATURE_FAMILIES)}"
)


def _comparison_metadata_columns() -> dict[str, object]:
    return {
        "experiment_result_id": EXPERIMENT_RESULT_ID,
        "run_artifact_key": RUN_ARTIFACT_KEY,
        "run_artifact_dir": str(RUN_ARTIFACT_DIR),
        "min_market_cap": int(MIN_MARKET_CAP),
        "base_train_end": BASE_TRAIN_END,
        "meta_oos_start": META_OOS_START,
        "score_threshold": float(SCORE_THRESHOLD),
        "simple_rule_cost_bps": float(SIMPLE_RULE_COST_BPS),
        "include_event_labels_in_oracle_labels": bool(INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS),
        "event_label_families": ",".join(EVENT_LABEL_FAMILIES if INCLUDE_EVENT_LABELS_IN_ORACLE_LABELS else ()),
        "include_technical_feature_families": bool(INCLUDE_TECHNICAL_FEATURE_FAMILIES),
        "include_family_score_rank_features": bool(INCLUDE_FAMILY_SCORE_RANK_FEATURES),
        "include_rank_mean_ensemble": bool(INCLUDE_RANK_MEAN_ENSEMBLE),
        "run_regime_year_feature": bool(RUN_REGIME_YEAR_FEATURE),
        "random_seed": int(RANDOM_SEED),
    }


def _write_experiment_comparison_table(name: str, frame: pd.DataFrame, *, key_columns: list[str] | None = None) -> None:
    if frame is None or frame.empty:
        return
    out = frame.copy()
    metadata = _comparison_metadata_columns()
    for column, value in metadata.items():
        out.insert(0, column, value)
    path = EXPERIMENT_COMPARISON_DIR / f"{name}.parquet"
    if path.exists():
        existing = pd.read_parquet(path)
        combined = pd.concat([existing, out], ignore_index=True, sort=False)
    else:
        combined = out
    dedupe_keys = ["experiment_result_id", *(key_columns or [])]
    dedupe_keys = [column for column in dedupe_keys if column in combined.columns]
    if dedupe_keys:
        combined = combined.drop_duplicates(subset=dedupe_keys, keep="last")
    combined.to_parquet(path, index=False)
    print(f"[experiment-comparison] wrote {name}: rows={len(out):,} total_rows={len(combined):,} path={path}")

base_model_results.to_csv(RUN_ARTIFACT_DIR / "base_model_results.csv", index=False)
meta_model_results.to_csv(RUN_ARTIFACT_DIR / "meta_model_results.csv", index=False)
classification_comparison.to_csv(RUN_ARTIFACT_DIR / "classification_comparison.csv", index=False)
backtest_summary.to_csv(RUN_ARTIFACT_DIR / "backtest_summary.csv", index=False)
trading_performance_leaderboard.to_csv(RUN_ARTIFACT_DIR / "trading_performance_leaderboard.csv", index=False)
yearly_backtest_summary.to_csv(RUN_ARTIFACT_DIR / "yearly_backtest_summary.csv", index=False)
if "regime_year_importance_summary" in globals() and not regime_year_importance_summary.empty:
    regime_year_importance_summary.to_csv(RUN_ARTIFACT_DIR / "regime_year_importance_summary.csv", index=False)
symbol_beat_summary.to_csv(RUN_ARTIFACT_DIR / "symbol_beat_summary.csv", index=False)
symbol_beat_by_group.to_csv(RUN_ARTIFACT_DIR / "symbol_beat_by_group.csv", index=False)

if SAVE_LARGE_INTERMEDIATE_CSVS:
    comparison_strategy_scores.to_csv(RUN_ARTIFACT_DIR / "comparison_strategy_scores.csv", index=False)
    trade_log.to_csv(RUN_ARTIFACT_DIR / "trade_log.csv", index=False)
    symbol_level_backtesting_py.to_csv(RUN_ARTIFACT_DIR / "symbol_level_backtesting_py.csv", index=False)
else:
    print("Skipped large intermediate CSV writes; daily score chunks and symbol checkpoint CSVs are the resumable artifacts.")

_write_experiment_comparison_table(
    "backtest_summary",
    backtest_summary,
    key_columns=["strategy_source", "rule", "threshold", "cost_bps", "start"],
)
_write_experiment_comparison_table(
    "trading_performance_leaderboard",
    trading_performance_leaderboard,
    key_columns=["strategy_source", "rule", "threshold", "cost_bps", "start"],
)
_write_experiment_comparison_table(
    "yearly_backtest_summary",
    yearly_backtest_summary,
    key_columns=["strategy_source", "year"],
)
_write_experiment_comparison_table(
    "symbol_beat_summary",
    symbol_beat_summary,
    key_columns=["strategy_source"],
)
_write_experiment_comparison_table(
    "symbol_beat_by_group",
    symbol_beat_by_group,
    key_columns=["strategy_source", "group"],
)
if "anchored_wfo_stitched_summary" in globals():
    _write_experiment_comparison_table(
        "anchored_wfo_stitched_summary",
        anchored_wfo_stitched_summary,
        key_columns=["strategy_source", "top_k"],
    )
if "anchored_wfo_fold_summary" in globals():
    _write_experiment_comparison_table(
        "anchored_wfo_fold_summary",
        anchored_wfo_fold_summary,
        key_columns=["forward_year", "strategy_source", "top_k"],
    )

meta_feature_manifest = {
    "experiment": "trading_app_v2_equity_meta_model_10b",
    "min_market_cap": MIN_MARKET_CAP,
    "base_train_end": BASE_TRAIN_END,
    "meta_oos_start": META_OOS_START,
    "score_threshold": SCORE_THRESHOLD,
    "simple_rule_cost_bps": SIMPLE_RULE_COST_BPS,
    "base_rf_params": BASE_RF_PARAMS,
    "meta_rf_params": META_RF_PARAMS,
    "meta_features": meta_feature_cols,
    "symbols": list(screened_equity_symbols),
    "feature_family_cache_dir": str(FEATURE_FAMILY_CACHE_DIR),
    "reuse_daily_score_chunks": REUSE_DAILY_SCORE_CHUNKS,
    "reuse_family_score_cache": REUSE_FAMILY_SCORE_CACHE,
    "rebuild_family_score_cache": REBUILD_FAMILY_SCORE_CACHE,
    "family_score_cache_dir": str(FAMILY_SCORE_CACHE_DIR),
    "daily_score_chunk_dir": str(RUN_ARTIFACT_DIR / ("daily_score_chunks_rank_features" if INCLUDE_FAMILY_SCORE_RANK_FEATURES else "daily_score_chunks")),
    "daily_family_score_cache_pattern": str(FAMILY_SCORE_CACHE_DIR / "daily_family_scores_{year}.parquet"),
    "base_oracle_family_score_cache": str(FAMILY_SCORE_CACHE_DIR / "base_oracle_family_scores.parquet"),
    "experiment_comparison_dir": str(EXPERIMENT_COMPARISON_DIR),
    "experiment_result_id": EXPERIMENT_RESULT_ID,
    "run_symbol_level_backtesting_py": RUN_SYMBOL_LEVEL_BACKTESTING_PY,
    "run_anchored_wfo": RUN_ANCHORED_WFO,
    "anchored_wfo_top_k_values": ANCHORED_WFO_TOP_K_VALUES,
    "include_family_strategy_wfo": INCLUDE_FAMILY_STRATEGY_WFO,
    "anchored_wfo_output_name": ANCHORED_WFO_OUTPUT_NAME,
    "include_family_score_rank_features": INCLUDE_FAMILY_SCORE_RANK_FEATURES,
    "include_rank_mean_ensemble": INCLUDE_RANK_MEAN_ENSEMBLE,
    "ensemble_strategy_sources": list(ENSEMBLE_STRATEGY_SOURCES),
    "include_technical_feature_families": INCLUDE_TECHNICAL_FEATURE_FAMILIES,
    "technical_strategy_sources": list(TECHNICAL_STRATEGY_SOURCES if INCLUDE_TECHNICAL_FEATURE_FAMILIES else ()),
    "experiment_strategy_sources": list(EXPERIMENT_STRATEGY_SOURCES),
    "save_large_intermediate_csvs": SAVE_LARGE_INTERMEDIATE_CSVS,
    "rebuild_feature_family_cache": REBUILD_FEATURE_FAMILY_CACHE,
    "run_regime_year_feature": RUN_REGIME_YEAR_FEATURE,
    "regime_year_feature": REGIME_YEAR_FEATURE if RUN_REGIME_YEAR_FEATURE else None,
    "storage": storage.as_dict(),
}
(RUN_ARTIFACT_DIR / "metadata.json").write_text(json.dumps(meta_feature_manifest, indent=2, default=str), encoding="utf-8")
with (RUN_ARTIFACT_DIR / "meta_classifier.pkl").open("wb") as handle:
    pickle.dump(meta_classifier, handle, protocol=pickle.HIGHEST_PROTOCOL)
print(f"Saved equity meta-model experiment artifacts to {RUN_ARTIFACT_DIR}")
